In [ ]:
# =============================================================================
# Evaluate fix rates and resample WTD data from approximately 3-hour fixes
# to FOUR GPS pings per day (~6-hour intervals)
#
# IMPORTANT:
#     4 pings/day = 24 hours / 4 = 6-hour target interval
#
# The actual observed GPS timestamp is retained for movement calculations.
# The artificial target timestamp is used only to define the intended
# four-pings-per-day sampling schedule.
# =============================================================================


from pathlib import Path

import numpy as np
import pandas as pd


# =============================================================================
# 1. FILE PATHS
# =============================================================================

BASE_PATH = Path(
    "<LOCAL_PATH>"
)

DATASETS = {

    "(Train)":
        "CWD_negative_training_with_seasons2_AKDE_FIXED_TRIMMED_150.csv",

    "(Test)":
        "CWD_positive_testing_with_seasons2_AKDE_FIXED_TRIMMED_150.csv"
}


# Output directory
OUTPUT_DIR = (
    BASE_PATH /
    "four_pings_per_day_resampled"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. USER SETTINGS
# =============================================================================

ID_COL = "id"

DATE_COL = "date"

STATUS_COL = "cwd_status"


# -----------------------------------------------------------------------------
# FOUR PINGS PER DAY
# -----------------------------------------------------------------------------

PINGS_PER_DAY = 4


# 24 hours / 4 pings = one target ping every 6 hours
TARGET_INTERVAL_HOURS = (
    24 / PINGS_PER_DAY
)


# Original data are nominally ~3-hour data.
#
# +/- 1.5 hours corresponds to half of the original 3-hour sampling interval.
MATCH_TOLERANCE_HOURS = 1.5


EXPECTED_FIXES_PER_DAY = (
    PINGS_PER_DAY
)


print(
    "\n"
    + "=" * 80
)

print(
    "TARGET SAMPLING DESIGN"
)

print(
    "=" * 80
)

print(
    f"Target pings per day:          "
    f"{PINGS_PER_DAY}"
)

print(
    f"Target interval:               "
    f"{TARGET_INTERVAL_HOURS:.1f} hours"
)

print(
    f"Nearest-fix matching tolerance:"
    f" +/- {MATCH_TOLERANCE_HOURS:.1f} hours"
)


# =============================================================================
# 3. BASIC CLEANING
# =============================================================================

def clean_datetime_data(
    df,
    label
):
    """
    Standardize ID and datetime columns and remove unusable records.
    """

    df = df.copy()


    required_columns = [
        ID_COL,
        DATE_COL
    ]


    missing_columns = [

        col

        for col in required_columns

        if col not in df.columns
    ]


    if missing_columns:

        raise ValueError(

            f"{label} is missing required columns: "
            f"{missing_columns}"
        )


    original_n = len(df)


    # -------------------------------------------------------------------------
    # Preserve IDs as strings
    # -------------------------------------------------------------------------

    df[ID_COL] = (
        df[ID_COL]
        .astype(str)
        .str.strip()
    )


    # -------------------------------------------------------------------------
    # Convert timestamps
    # -------------------------------------------------------------------------

    df[DATE_COL] = pd.to_datetime(

        df[DATE_COL],

        errors="coerce"
    )


    invalid_dates = (
        df[DATE_COL]
        .isna()
        .sum()
    )


    missing_ids = (

        df[ID_COL].isna()

        |

        df[ID_COL].eq("")

        |

        df[ID_COL].str.lower().eq("nan")
    ).sum()


    # -------------------------------------------------------------------------
    # Remove unusable rows
    # -------------------------------------------------------------------------

    df = df.dropna(
        subset=[DATE_COL]
    )


    df = df[

        df[ID_COL].notna()

        &

        df[ID_COL].ne("")

        &

        df[ID_COL].str.lower().ne("nan")

    ].copy()


    # -------------------------------------------------------------------------
    # Remove duplicate ID/timestamp combinations
    # -------------------------------------------------------------------------

    duplicate_count = (

        df.duplicated(

            subset=[
                ID_COL,
                DATE_COL
            ],

            keep="first"

        ).sum()
    )


    df = df.drop_duplicates(

        subset=[
            ID_COL,
            DATE_COL
        ],

        keep="first"
    )


    df = (

        df.sort_values(
            [
                ID_COL,
                DATE_COL
            ]
        )

        .reset_index(
            drop=True
        )
    )


    print(
        f"\n{'=' * 80}"
    )

    print(
        f"{label}: DATETIME CLEANING"
    )

    print(
        f"{'=' * 80}"
    )


    print(
        f"Original rows:                "
        f"{original_n:,}"
    )

    print(
        f"Invalid timestamps:           "
        f"{invalid_dates:,}"
    )

    print(
        f"Missing/invalid IDs:          "
        f"{missing_ids:,}"
    )

    print(
        f"Duplicate ID/timestamps:      "
        f"{duplicate_count:,}"
    )

    print(
        f"Rows retained after cleaning: "
        f"{len(df):,}"
    )


    return df


# =============================================================================
# 4. ID, STATUS, AND FIX-COUNT SUMMARY
# =============================================================================

def print_unique_ids_and_status(
    df,
    label
):
    """
    Print unique IDs, CWD statuses, and total fix counts.
    """

    ids = sorted(

        df[ID_COL]
        .dropna()
        .unique()
    )


    print(
        f"\n{'=' * 80}"
    )

    print(
        f"{label}: ID AND STATUS SUMMARY"
    )

    print(
        f"{'=' * 80}"
    )


    print(
        f"\n{len(ids)} unique IDs:"
    )

    print(
        ids
    )


    # -------------------------------------------------------------------------
    # CWD status
    # -------------------------------------------------------------------------

    if STATUS_COL in df.columns:

        status_values = sorted(

            df[STATUS_COL]
            .dropna()
            .astype(str)
            .unique()
        )


        print(
            f"\nObserved status values: "
            f"{status_values}"
        )


        positive_ids = sorted(

            df.loc[

                df[STATUS_COL]
                .astype(str)
                .str.lower()
                .eq("positive"),

                ID_COL

            ].unique()
        )


        negative_ids = sorted(

            df.loc[

                df[STATUS_COL]
                .astype(str)
                .str.lower()
                .eq("negative"),

                ID_COL

            ].unique()
        )


        print(

            f"CWD-positive IDs "
            f"({len(positive_ids)}): "

            f"{positive_ids}"
        )


        print(

            f"CWD-negative IDs "
            f"({len(negative_ids)}): "

            f"{negative_ids}"
        )


    # -------------------------------------------------------------------------
    # Fix counts
    # -------------------------------------------------------------------------

    fix_counts = (

        df.groupby(ID_COL)

        .size()

        .sort_index()
    )


    print(
        "\nNumber of GPS fixes per ID:"
    )

    print(
        fix_counts.to_string()
    )


    print(
        "\nFix-count summary:"
    )

    print(
        f"  Minimum: "
        f"{fix_counts.min():,.0f}"
    )

    print(
        f"  Maximum: "
        f"{fix_counts.max():,.0f}"
    )

    print(
        f"  Median:  "
        f"{fix_counts.median():,.1f}"
    )

    print(
        f"  Mean:    "
        f"{fix_counts.mean():,.1f}"
    )


# =============================================================================
# 5. DETAILED FIX-RATE EDA
# =============================================================================

def summarize_fix_rate(
    df,
    label
):
    """
    Summarize tracking duration, time between observations,
    and fixes per calendar day.
    """

    working = (

        df.sort_values(
            [
                ID_COL,
                DATE_COL
            ]
        )

        .copy()
    )


    # -------------------------------------------------------------------------
    # Tracking duration
    # -------------------------------------------------------------------------

    tracking_summary = (

        working

        .groupby(ID_COL)[DATE_COL]

        .agg(

            first_fix="min",

            last_fix="max",

            total_fixes="size"
        )

        .reset_index()
    )


    tracking_summary["duration_days"] = (

        tracking_summary["last_fix"]

        -

        tracking_summary["first_fix"]

    ).dt.total_seconds() / 86_400


    # Inclusive approximate calendar coverage
    tracking_summary["inclusive_days"] = (

        tracking_summary["duration_days"]

        +

        1
    )


    tracking_summary[
        "fixes_per_tracking_day"
    ] = (

        tracking_summary["total_fixes"]

        /

        tracking_summary["inclusive_days"]
    )


    # -------------------------------------------------------------------------
    # Time interval between consecutive observations
    # -------------------------------------------------------------------------

    working["interval_hours"] = (

        working

        .groupby(ID_COL)[DATE_COL]

        .diff()

        .dt.total_seconds()

        .div(3600)
    )


    positive_intervals = working.loc[

        working["interval_hours"] > 0,

        "interval_hours"
    ]


    # -------------------------------------------------------------------------
    # Fixes per calendar day
    # -------------------------------------------------------------------------

    working["calendar_date"] = (

        working[DATE_COL]
        .dt.date
    )


    daily_counts = (

        working

        .groupby(
            [
                ID_COL,
                "calendar_date"
            ]
        )

        .size()

        .rename(
            "fixes_per_day"
        )

        .reset_index()
    )


    per_id_daily_summary = (

        daily_counts

        .groupby(ID_COL)[
            "fixes_per_day"
        ]

        .agg(

            number_of_observed_days="size",

            mean_fixes_per_day="mean",

            median_fixes_per_day="median",

            minimum_fixes_per_day="min",

            maximum_fixes_per_day="max"
        )

        .reset_index()
    )


    id_summary = (

        tracking_summary

        .merge(

            per_id_daily_summary,

            on=ID_COL,

            how="left"
        )
    )


    # -------------------------------------------------------------------------
    # Print
    # -------------------------------------------------------------------------

    print(
        f"\n{'=' * 80}"
    )

    print(
        f"{label}: FIX-RATE EDA"
    )

    print(
        f"{'=' * 80}"
    )


    print(

        f"Overall date range: "

        f"{working[DATE_COL].min()} "

        f"to "

        f"{working[DATE_COL].max()}"
    )


    print(
        "\nTracking duration across IDs:"
    )


    print(

        f"  Minimum: "

        f"{tracking_summary['duration_days'].min():.2f} days"
    )


    print(

        f"  Maximum: "

        f"{tracking_summary['duration_days'].max():.2f} days"
    )


    print(

        f"  Median:  "

        f"{tracking_summary['duration_days'].median():.2f} days"
    )


    print(

        f"  Mean:    "

        f"{tracking_summary['duration_days'].mean():.2f} days"
    )


    # -------------------------------------------------------------------------
    # Interval distribution
    # -------------------------------------------------------------------------

    if not positive_intervals.empty:

        print(
            "\nIntervals between consecutive fixes:"
        )


        print(

            f"  Minimum: "

            f"{positive_intervals.min():.2f} hours"
        )


        print(

            f"  25th percentile: "

            f"{positive_intervals.quantile(0.25):.2f} hours"
        )


        print(

            f"  Median:  "

            f"{positive_intervals.median():.2f} hours"
        )


        print(

            f"  Mean:    "

            f"{positive_intervals.mean():.2f} hours"
        )


        print(

            f"  75th percentile: "

            f"{positive_intervals.quantile(0.75):.2f} hours"
        )


        print(

            f"  Maximum: "

            f"{positive_intervals.max():.2f} hours"
        )


        approximately_3_hour = (

            positive_intervals

            .between(
                2.5,
                3.5
            )

            .mean()

            *

            100
        )


        approximately_6_hour = (

            positive_intervals

            .between(
                5.5,
                6.5
            )

            .mean()

            *

            100
        )


        print(

            f"  Intervals between "
            f"2.5 and 3.5 hours: "

            f"{approximately_3_hour:.1f}%"
        )


        print(

            f"  Intervals between "
            f"5.5 and 6.5 hours: "

            f"{approximately_6_hour:.1f}%"
        )


    # -------------------------------------------------------------------------
    # Daily fix distribution
    # -------------------------------------------------------------------------

    print(
        "\nDaily fix-count distribution across all ID-days:"
    )


    print(

        daily_counts[
            "fixes_per_day"
        ]

        .describe()

        .to_string()
    )


    print(
        "\nMean daily fix rate across IDs:"
    )


    print(

        f"  Minimum: "

        f"{per_id_daily_summary['mean_fixes_per_day'].min():.2f}"
    )


    print(

        f"  Maximum: "

        f"{per_id_daily_summary['mean_fixes_per_day'].max():.2f}"
    )


    print(

        f"  Median:  "

        f"{per_id_daily_summary['mean_fixes_per_day'].median():.2f}"
    )


    print(

        f"  Mean:    "

        f"{per_id_daily_summary['mean_fixes_per_day'].mean():.2f}"
    )


    # -------------------------------------------------------------------------
    # Per-ID summary
    # -------------------------------------------------------------------------

    print(
        "\nPer-ID sampling summary:"
    )


    columns_to_print = [

        ID_COL,

        "total_fixes",

        "duration_days",

        "fixes_per_tracking_day",

        "mean_fixes_per_day",

        "median_fixes_per_day",

        "minimum_fixes_per_day",

        "maximum_fixes_per_day"
    ]


    print(

        id_summary[
            columns_to_print
        ]

        .round(2)

        .to_string(
            index=False
        )
    )


    return {

        "tracking_summary":
            tracking_summary,

        "daily_counts":
            daily_counts,

        "id_summary":
            id_summary,

        "interval_data":
            working[
                [
                    ID_COL,
                    DATE_COL,
                    "interval_hours"
                ]
            ].copy()
    }


# =============================================================================
# 6. RESAMPLE ONE DEER TO FOUR PINGS PER DAY
# =============================================================================

def resample_single_id_to_four_pings(
    group,
    pings_per_day=4,
    tolerance_hours=1.5
):
    """
    Resample one animal to FOUR GPS fixes per day.

    Four pings/day corresponds to one target fix every six hours.

    The target grid is anchored to the animal's first actual GPS fix rather
    than forcing all individuals onto midnight-based times.

    Example
    -------

    If the first GPS observation is:

        02:10

    then the intended six-hour target schedule becomes approximately:

        02:10
        08:10
        14:10
        20:10

    each full day.

    IMPORTANT
    ---------

    `date`
        remains the REAL observed GPS timestamp.

    `original_fix_time`
        is also the real observed GPS timestamp.

    `target_four_ping_time`
        is the intended six-hour target time.

    Therefore movement calculations continue to use the true observation
    times rather than artificial target times.
    """

    interval_hours = (
        24 / pings_per_day
    )


    group = (

        group

        .sort_values(DATE_COL)

        .drop_duplicates(
            subset=[DATE_COL],
            keep="first"
        )

        .copy()
    )


    if group.empty:

        return group


    animal_id = (
        group[ID_COL]
        .iloc[0]
    )


    # -------------------------------------------------------------------------
    # Guarantee retention of IDs with only one observation
    # -------------------------------------------------------------------------

    if len(group) == 1:

        result = (
            group.copy()
        )


        result[
            "original_fix_time"
        ] = result[DATE_COL]


        result[
            "target_four_ping_time"
        ] = result[DATE_COL]


        result[
            "target_ping_number"
        ] = 1


        result[
            "time_offset_minutes"
        ] = 0.0


        result[
            "target_pings_per_day"
        ] = pings_per_day


        result[
            "resampling_interval_hours"
        ] = interval_hours


        result[
            "resampling_tolerance_hours"
        ] = tolerance_hours


        result[
            "resampling_method"
        ] = "single_fix_retained"


        return result.reset_index(
            drop=True
        )


    # -------------------------------------------------------------------------
    # Define tracking range
    # -------------------------------------------------------------------------

    first_time = (
        group[DATE_COL]
        .min()
    )


    last_time = (
        group[DATE_COL]
        .max()
    )


    # -------------------------------------------------------------------------
    # Create continuous six-hour target grid
    #
    # Because 24 / 6 = 4, every COMPLETE 24-hour period has four targets.
    #
    # Only the first and final partial tracking days may have fewer than four
    # available targets.
    # -------------------------------------------------------------------------

    target_times = pd.date_range(

        start=first_time,

        end=last_time,

        freq=pd.Timedelta(
            hours=interval_hours
        )
    )


    target_grid = pd.DataFrame({

        "target_four_ping_time":
            target_times
    })


    if target_grid.empty:

        result = (
            group.iloc[[0]]
            .copy()
        )


        result[
            "original_fix_time"
        ] = result[DATE_COL]


        result[
            "target_four_ping_time"
        ] = result[DATE_COL]


        result[
            "target_ping_number"
        ] = 1


        result[
            "time_offset_minutes"
        ] = 0.0


        result[
            "target_pings_per_day"
        ] = pings_per_day


        result[
            "resampling_interval_hours"
        ] = interval_hours


        result[
            "resampling_tolerance_hours"
        ] = tolerance_hours


        result[
            "resampling_method"
        ] = "forced_first_fix"


        return result.reset_index(
            drop=True
        )


    # -------------------------------------------------------------------------
    # Assign target calendar date
    # -------------------------------------------------------------------------

    target_grid[
        "target_calendar_date"
    ] = (

        target_grid[
            "target_four_ping_time"
        ]

        .dt.date
    )


    # -------------------------------------------------------------------------
    # Number each target within its target calendar day
    # -------------------------------------------------------------------------

    target_grid[
        "target_ping_number"
    ] = (

        target_grid

        .groupby(
            "target_calendar_date"
        )

        .cumcount()

        +

        1
    )


    # -------------------------------------------------------------------------
    # Expected number of target slots occurring on each calendar day
    #
    # Full interior days should have 4.
    # Edge days may contain fewer because tracking begins/ends mid-day.
    # -------------------------------------------------------------------------

    target_grid[
        "expected_target_slots_that_day"
    ] = (

        target_grid

        .groupby(
            "target_calendar_date"
        )[
            "target_four_ping_time"
        ]

        .transform(
            "size"
        )
    )


    # -------------------------------------------------------------------------
    # Prepare source observations
    # -------------------------------------------------------------------------

    source = (
        group.copy()
    )


    source[
        "original_fix_time"
    ] = source[DATE_COL]


    source[
        "_source_row_id"
    ] = np.arange(
        len(source)
    )


    source = (

        source

        .sort_values(
            "original_fix_time"
        )
    )


    target_grid = (

        target_grid

        .sort_values(
            "target_four_ping_time"
        )
    )


    # -------------------------------------------------------------------------
    # Match each six-hour target to nearest actual observation
    # -------------------------------------------------------------------------

    matched = pd.merge_asof(

        target_grid,

        source,

        left_on=
            "target_four_ping_time",

        right_on=
            "original_fix_time",

        direction=
            "nearest",

        tolerance=
            pd.Timedelta(
                hours=tolerance_hours
            )
    )


    # -------------------------------------------------------------------------
    # Keep only targets for which an actual observation was found
    # -------------------------------------------------------------------------

    matched = (

        matched

        .dropna(
            subset=[
                "_source_row_id"
            ]
        )

        .copy()
    )


    resampling_method = (
        "nearest_with_tolerance"
    )


    # -------------------------------------------------------------------------
    # If absolutely no target matches were found for this animal,
    # repeat without tolerance so that the ID is not lost.
    # -------------------------------------------------------------------------

    if matched.empty:

        matched = pd.merge_asof(

            target_grid,

            source,

            left_on=
                "target_four_ping_time",

            right_on=
                "original_fix_time",

            direction=
                "nearest"
        )


        matched = (

            matched

            .dropna(
                subset=[
                    "_source_row_id"
                ]
            )

            .copy()
        )


        resampling_method = (
            "nearest_without_tolerance"
        )


    # -------------------------------------------------------------------------
    # Calculate deviation from intended target time
    # -------------------------------------------------------------------------

    if not matched.empty:

        matched[
            "time_offset_minutes"
        ] = (

            matched[
                "original_fix_time"
            ]

            -

            matched[
                "target_four_ping_time"
            ]

        ).dt.total_seconds().abs() / 60


        # ---------------------------------------------------------------------
        # Prevent one observed GPS fix from filling two target slots.
        #
        # If an observation happens to be nearest to multiple targets,
        # retain only the closest target.
        # ---------------------------------------------------------------------

        matched = (

            matched

            .sort_values(

                [
                    "_source_row_id",
                    "time_offset_minutes"
                ]
            )

            .drop_duplicates(

                subset=[
                    "_source_row_id"
                ],

                keep="first"
            )

            .sort_values(
                "target_four_ping_time"
            )

            .copy()
        )


    # -------------------------------------------------------------------------
    # Absolute fallback
    #
    # Original data are approximately 3-hour.
    # Every second observation therefore approximates 6-hour sampling.
    # -------------------------------------------------------------------------

    if matched.empty:

        matched = (

            group

            .iloc[::2]

            .copy()
        )


        if matched.empty:

            matched = (

                group

                .iloc[[0]]

                .copy()
            )


        matched[
            "original_fix_time"
        ] = matched[DATE_COL]


        matched[
            "target_four_ping_time"
        ] = matched[DATE_COL]


        matched[
            "target_calendar_date"
        ] = (

            matched[
                "target_four_ping_time"
            ]

            .dt.date
        )


        matched[
            "target_ping_number"
        ] = (

            matched

            .groupby(
                "target_calendar_date"
            )

            .cumcount()

            +

            1
        )


        matched[
            "expected_target_slots_that_day"
        ] = (

            matched

            .groupby(
                "target_calendar_date"
            )[DATE_COL]

            .transform(
                "size"
            )
        )


        matched[
            "time_offset_minutes"
        ] = 0.0


        resampling_method = (
            "every_second_fix_fallback"
        )


    # -------------------------------------------------------------------------
    # Preserve ID
    # -------------------------------------------------------------------------

    matched[ID_COL] = (
        animal_id
    )


    # -------------------------------------------------------------------------
    # CRITICAL:
    #
    # Analytical date remains the REAL GPS observation time.
    #
    # We DO NOT replace it with the artificial target time.
    # -------------------------------------------------------------------------

    matched[DATE_COL] = (

        matched[
            "original_fix_time"
        ]
    )


    # -------------------------------------------------------------------------
    # Store resampling metadata
    # -------------------------------------------------------------------------

    matched[
        "target_pings_per_day"
    ] = pings_per_day


    matched[
        "resampling_interval_hours"
    ] = interval_hours


    matched[
        "resampling_tolerance_hours"
    ] = tolerance_hours


    matched[
        "resampling_method"
    ] = resampling_method


    matched = matched.drop(

        columns=[
            "_source_row_id"
        ],

        errors="ignore"
    )


    return matched.reset_index(
        drop=True
    )


# =============================================================================
# 7. RESAMPLE COMPLETE DATASET TO FOUR PINGS PER DAY
# =============================================================================

def resample_dataset_to_four_pings(
    df,
    label,
    pings_per_day=4,
    tolerance_hours=1.5
):
    """
    Apply four-pings-per-day resampling separately to each deer.

    All original IDs are explicitly checked to ensure that none disappear.
    """

    interval_hours = (
        24 / pings_per_day
    )


    original_ids = set(

        df[ID_COL]

        .dropna()

        .astype(str)

        .unique()
    )


    resampled_groups = []


    # -------------------------------------------------------------------------
    # Process each animal independently
    # -------------------------------------------------------------------------

    for animal_id, group in df.groupby(
        ID_COL,
        sort=True
    ):

        resampled_group = (

            resample_single_id_to_four_pings(

                group=group,

                pings_per_day=
                    pings_per_day,

                tolerance_hours=
                    tolerance_hours
            )
        )


        # ---------------------------------------------------------------------
        # Final safety check so no ID disappears
        # ---------------------------------------------------------------------

        if resampled_group.empty:

            print(

                f"Warning: fallback required for "
                f"ID {animal_id}. "

                f"Retaining first observation."
            )


            resampled_group = (

                group

                .sort_values(
                    DATE_COL
                )

                .iloc[[0]]

                .copy()
            )


            resampled_group[
                "original_fix_time"
            ] = (

                resampled_group[
                    DATE_COL
                ]
            )


            resampled_group[
                "target_four_ping_time"
            ] = (

                resampled_group[
                    DATE_COL
                ]
            )


            resampled_group[
                "target_calendar_date"
            ] = (

                resampled_group[
                    DATE_COL
                ]

                .dt.date
            )


            resampled_group[
                "target_ping_number"
            ] = 1


            resampled_group[
                "expected_target_slots_that_day"
            ] = 1


            resampled_group[
                "time_offset_minutes"
            ] = 0.0


            resampled_group[
                "target_pings_per_day"
            ] = pings_per_day


            resampled_group[
                "resampling_interval_hours"
            ] = interval_hours


            resampled_group[
                "resampling_tolerance_hours"
            ] = tolerance_hours


            resampled_group[
                "resampling_method"
            ] = "forced_first_fix"


        resampled_groups.append(
            resampled_group
        )


    if not resampled_groups:

        raise ValueError(

            f"No observations were retained "
            f"for {label}."
        )


    # -------------------------------------------------------------------------
    # Combine IDs
    # -------------------------------------------------------------------------

    resampled = pd.concat(

        resampled_groups,

        ignore_index=True
    )


    resampled[ID_COL] = (

        resampled[ID_COL]

        .astype(str)

        .str.strip()
    )


    resampled = (

        resampled

        .sort_values(
            [
                ID_COL,
                DATE_COL
            ]
        )

        .reset_index(
            drop=True
        )
    )


    # -------------------------------------------------------------------------
    # Verify ID retention
    # -------------------------------------------------------------------------

    resampled_ids = set(

        resampled[ID_COL]

        .dropna()

        .astype(str)

        .unique()
    )


    missing_ids = sorted(

        original_ids

        -

        resampled_ids
    )


    added_ids = sorted(

        resampled_ids

        -

        original_ids
    )


    print(
        f"\n{'=' * 80}"
    )

    print(
        f"{label}: ID RETENTION CHECK"
    )

    print(
        f"{'=' * 80}"
    )


    print(

        f"Original unique IDs:  "
        f"{len(original_ids)}"
    )


    print(

        f"Resampled unique IDs: "
        f"{len(resampled_ids)}"
    )


    if missing_ids:

        raise RuntimeError(

            "The following IDs were unexpectedly lost: "

            f"{missing_ids}"
        )

    else:

        print(
            "All original IDs were retained."
        )


    if added_ids:

        raise RuntimeError(

            "Unexpected IDs appeared in output: "

            f"{added_ids}"
        )


    # -------------------------------------------------------------------------
    # Fix-count comparison
    # -------------------------------------------------------------------------

    original_counts = (

        df

        .groupby(ID_COL)

        .size()

        .rename(
            "original_fixes"
        )
    )


    resampled_counts = (

        resampled

        .groupby(ID_COL)

        .size()

        .rename(
            "four_ping_fixes"
        )
    )


    comparison = pd.concat(

        [
            original_counts,
            resampled_counts
        ],

        axis=1

    ).fillna(0)


    comparison[
        "original_fixes"
    ] = (

        comparison[
            "original_fixes"
        ]

        .astype(int)
    )


    comparison[
        "four_ping_fixes"
    ] = (

        comparison[
            "four_ping_fixes"
        ]

        .astype(int)
    )


    comparison[
        "retained_percent"
    ] = (

        comparison[
            "four_ping_fixes"
        ]

        /

        comparison[
            "original_fixes"
        ]

        *

        100
    )


    print(
        f"\n{'=' * 80}"
    )

    print(
        f"{label}: FOUR-PINGS-PER-DAY RESAMPLING RESULTS"
    )

    print(
        f"{'=' * 80}"
    )


    print(

        f"Original rows:          "
        f"{len(df):,}"
    )


    print(

        f"Four-ping rows:         "
        f"{len(resampled):,}"
    )


    print(

        f"Overall retention:      "

        f"{len(resampled) / len(df) * 100:.1f}%"
    )


    print(
        "\nRetention by ID:"
    )


    print(

        comparison

        .round(1)

        .to_string()
    )


    # -------------------------------------------------------------------------
    # Matching method summary
    # -------------------------------------------------------------------------

    if "resampling_method" in resampled.columns:

        method_summary = (

            resampled

            .groupby(
                "resampling_method"
            )

            .agg(

                rows=(
                    ID_COL,
                    "size"
                ),

                unique_ids=(
                    ID_COL,
                    "nunique"
                )
            )

            .reset_index()
        )


        print(
            "\nResampling methods used:"
        )


        print(

            method_summary

            .to_string(
                index=False
            )
        )


    return (

        resampled,

        comparison.reset_index()
    )


# =============================================================================
# 8. EXPLICIT TRAIN/TEST ID VERIFICATION
# =============================================================================

def verify_no_ids_lost(
    original_df,
    resampled_df,
    label
):
    """
    Compare original and resampled ID sets.
    """

    original_ids = set(

        original_df[ID_COL]

        .astype(str)

        .str.strip()

        .unique()
    )


    resampled_ids = set(

        resampled_df[ID_COL]

        .astype(str)

        .str.strip()

        .unique()
    )


    lost_ids = sorted(

        original_ids

        -

        resampled_ids
    )


    retained_ids = sorted(

        original_ids

        &

        resampled_ids
    )


    print(
        f"\n{'=' * 80}"
    )

    print(
        f"{label}: FINAL ID VERIFICATION"
    )

    print(
        f"{'=' * 80}"
    )


    print(
        f"Original IDs: "
        f"{len(original_ids)}"
    )


    print(
        f"Retained IDs: "
        f"{len(retained_ids)}"
    )


    print(
        f"Lost IDs:     "
        f"{len(lost_ids)}"
    )


    if lost_ids:

        print(
            f"Lost ID list: "
            f"{lost_ids}"
        )


        raise AssertionError(

            f"{label} lost "
            f"{len(lost_ids)} IDs "
            f"during resampling."
        )


    print(
        "Verification passed: no IDs were lost."
    )


# =============================================================================
# 8A. VERIFY FOUR-PINGS-PER-DAY OUTPUT
# =============================================================================

def evaluate_four_ping_output(
    df,
    label
):
    """
    Verify the sampling rate in the new four-pings-per-day dataset.

    IMPORTANT:

    Actual observation timestamps are used to calculate real intervals.

    Target timestamps are used to calculate target fixes per day.

    This prevents observations slightly before/after midnight from
    incorrectly moving into another sampling day.
    """

    check = (

        df

        .sort_values(
            [
                ID_COL,
                DATE_COL
            ]
        )

        .copy()
    )


    # -------------------------------------------------------------------------
    # Actual interval between retained GPS observations
    # -------------------------------------------------------------------------

    check[
        "interval_hours"
    ] = (

        check

        .groupby(ID_COL)[
            DATE_COL
        ]

        .diff()

        .dt.total_seconds()

        .div(3600)
    )


    valid_intervals = check.loc[

        check[
            "interval_hours"
        ] > 0,

        "interval_hours"
    ]


    # -------------------------------------------------------------------------
    # Verify target time exists
    # -------------------------------------------------------------------------

    if (
        "target_four_ping_time"
        not in check.columns
    ):

        raise ValueError(

            "target_four_ping_time is missing "
            "from the resampled dataframe."
        )


    check[
        "target_four_ping_time"
    ] = pd.to_datetime(

        check[
            "target_four_ping_time"
        ],

        errors="coerce"
    )


    # -------------------------------------------------------------------------
    # Define target sampling date
    # -------------------------------------------------------------------------

    check[
        "target_calendar_date"
    ] = (

        check[
            "target_four_ping_time"
        ]

        .dt.date
    )


    # -------------------------------------------------------------------------
    # Count matched observations per target sampling day
    # -------------------------------------------------------------------------

    daily_counts = (

        check

        .groupby(
            [
                ID_COL,
                "target_calendar_date"
            ]
        )

        .agg(

            fixes_per_day=(
                DATE_COL,
                "size"
            ),

            expected_target_slots=(
                "expected_target_slots_that_day",
                "max"
            )
        )

        .reset_index()
    )


    # -------------------------------------------------------------------------
    # Identify complete target days
    #
    # First and final tracking dates may contain fewer than four intended
    # target slots simply because tracking begins or ends partway through
    # the day.
    # -------------------------------------------------------------------------

    daily_counts[
        "full_target_day"
    ] = (

        daily_counts[
            "expected_target_slots"
        ]

        ==

        EXPECTED_FIXES_PER_DAY
    )


    full_days = daily_counts.loc[

        daily_counts[
            "full_target_day"
        ]

    ].copy()


    # -------------------------------------------------------------------------
    # Metrics across ALL observed target days
    # -------------------------------------------------------------------------

    exactly_four_all = (

        daily_counts[
            "fixes_per_day"
        ]

        ==

        EXPECTED_FIXES_PER_DAY

    ).mean() * 100


    at_least_four_all = (

        daily_counts[
            "fixes_per_day"
        ]

        >=

        EXPECTED_FIXES_PER_DAY

    ).mean() * 100


    fewer_than_four_all = (

        daily_counts[
            "fixes_per_day"
        ]

        <

        EXPECTED_FIXES_PER_DAY

    ).mean() * 100


    # -------------------------------------------------------------------------
    # Metrics on COMPLETE target days only
    # -------------------------------------------------------------------------

    if not full_days.empty:

        exactly_four_full = (

            full_days[
                "fixes_per_day"
            ]

            ==

            EXPECTED_FIXES_PER_DAY

        ).mean() * 100


        fewer_than_four_full = (

            full_days[
                "fixes_per_day"
            ]

            <

            EXPECTED_FIXES_PER_DAY

        ).mean() * 100


    else:

        exactly_four_full = np.nan

        fewer_than_four_full = np.nan


    # -------------------------------------------------------------------------
    # Print verification
    # -------------------------------------------------------------------------

    print(
        f"\n{'=' * 80}"
    )

    print(

        f"{label}: VERIFICATION OF "
        f"FOUR-PINGS-PER-DAY OUTPUT"
    )

    print(
        f"{'=' * 80}"
    )


    print(

        f"Target sampling design: "
        f"{EXPECTED_FIXES_PER_DAY} pings/day"
    )


    print(

        f"Target interval: "
        f"{TARGET_INTERVAL_HOURS:.1f} hours"
    )


    # -------------------------------------------------------------------------
    # Actual interval diagnostics
    # -------------------------------------------------------------------------

    if not valid_intervals.empty:

        print(

            "\nActual intervals between "
            "retained GPS observations:"
        )


        print(

            f"  Minimum: "

            f"{valid_intervals.min():.2f} hours"
        )


        print(

            f"  Median:  "

            f"{valid_intervals.median():.2f} hours"
        )


        print(

            f"  Mean:    "

            f"{valid_intervals.mean():.2f} hours"
        )


        print(

            f"  Maximum: "

            f"{valid_intervals.max():.2f} hours"
        )


        approximately_six_hour = (

            valid_intervals

            .between(
                5.5,
                6.5
            )

            .mean()

            *

            100
        )


        print(

            f"  Intervals between "
            f"5.5 and 6.5 hours: "

            f"{approximately_six_hour:.1f}%"
        )


    # -------------------------------------------------------------------------
    # Daily counts
    # -------------------------------------------------------------------------

    print(
        "\nTarget fixes per sampling day:"
    )


    print(

        daily_counts[
            "fixes_per_day"
        ]

        .describe()

        .to_string()
    )


    print(

        f"\nALL target days:"
    )


    print(

        f"  Exactly "
        f"{EXPECTED_FIXES_PER_DAY} fixes: "

        f"{exactly_four_all:.1f}%"
    )


    print(

        f"  At least "
        f"{EXPECTED_FIXES_PER_DAY} fixes: "

        f"{at_least_four_all:.1f}%"
    )


    print(

        f"  Fewer than "
        f"{EXPECTED_FIXES_PER_DAY} fixes: "

        f"{fewer_than_four_all:.1f}%"
    )


    # -------------------------------------------------------------------------
    # Full target days
    # -------------------------------------------------------------------------

    print(

        f"\nComplete target days only "
        f"(days where four target slots existed):"
    )


    print(

        f"  Number of complete target ID-days: "
        f"{len(full_days):,}"
    )


    if not full_days.empty:

        print(

            f"  Exactly four retained fixes: "
            f"{exactly_four_full:.1f}%"
        )


        print(

            f"  Missing >=1 target fix:      "
            f"{fewer_than_four_full:.1f}%"
        )


    # -------------------------------------------------------------------------
    # Per-ID daily summary
    # -------------------------------------------------------------------------

    id_daily_summary = (

        daily_counts

        .groupby(ID_COL)

        .agg(

            observed_target_days=(
                "fixes_per_day",
                "size"
            ),

            mean_fixes_per_day=(
                "fixes_per_day",
                "mean"
            ),

            median_fixes_per_day=(
                "fixes_per_day",
                "median"
            ),

            minimum_fixes_per_day=(
                "fixes_per_day",
                "min"
            ),

            maximum_fixes_per_day=(
                "fixes_per_day",
                "max"
            ),

            full_target_days=(
                "full_target_day",
                "sum"
            )
        )

        .reset_index()
    )


    # -------------------------------------------------------------------------
    # Number of days containing exactly four matched observations
    # -------------------------------------------------------------------------

    four_day_counts = (

        daily_counts

        .assign(

            is_four=lambda x:

                x[
                    "fixes_per_day"
                ]

                ==

                EXPECTED_FIXES_PER_DAY
        )

        .groupby(ID_COL)[
            "is_four"
        ]

        .sum()

        .rename(
            "days_with_four_fixes"
        )

        .reset_index()
    )


    id_daily_summary = (

        id_daily_summary

        .merge(

            four_day_counts,

            on=ID_COL,

            how="left"
        )
    )


    id_daily_summary[
        "percent_days_with_four"
    ] = (

        id_daily_summary[
            "days_with_four_fixes"
        ]

        /

        id_daily_summary[
            "observed_target_days"
        ]

        *

        100
    )


    print(
        "\nPer-ID daily fix summary after resampling:"
    )


    print(

        id_daily_summary

        .round(2)

        .to_string(
            index=False
        )
    )


    return (

        daily_counts,

        id_daily_summary
    )


# =============================================================================
# 9. MAIN PROCESSING LOOP
# =============================================================================

all_original_data = {}

all_four_ping_data = {}

all_retention_tables = {}


for label, file_name in DATASETS.items():


    input_path = (

        BASE_PATH /
        file_name
    )


    print(
        f"\n\n{'#' * 80}"
    )

    print(
        f"PROCESSING {label.upper()} DATASET"
    )

    print(
        f"Input: {input_path}"
    )

    print(
        f"{'#' * 80}"
    )


    # -------------------------------------------------------------------------
    # Confirm file exists
    # -------------------------------------------------------------------------

    if not input_path.exists():

        raise FileNotFoundError(

            f"File not found: "
            f"{input_path}"
        )


    # -------------------------------------------------------------------------
    # Load original data
    # -------------------------------------------------------------------------

    original_df = pd.read_csv(
        input_path
    )


    # -------------------------------------------------------------------------
    # Clean original data
    # -------------------------------------------------------------------------

    original_df = clean_datetime_data(

        original_df,

        label=
            f"{label} original"
    )


    # -------------------------------------------------------------------------
    # ID/status summary
    # -------------------------------------------------------------------------

    print_unique_ids_and_status(

        original_df,

        label=
            f"{label} original"
    )


    # -------------------------------------------------------------------------
    # Original sampling EDA
    # -------------------------------------------------------------------------

    original_eda = summarize_fix_rate(

        original_df,

        label=
            f"{label} original"
    )


    # -------------------------------------------------------------------------
    # Create FOUR-PINGS-PER-DAY dataset
    # -------------------------------------------------------------------------

    four_ping_df, retention_table = (

        resample_dataset_to_four_pings(

            df=
                original_df,

            label=
                label,

            pings_per_day=
                PINGS_PER_DAY,

            tolerance_hours=
                MATCH_TOLERANCE_HOURS
        )
    )


    # -------------------------------------------------------------------------
    # Explicit ID retention check
    # -------------------------------------------------------------------------

    verify_no_ids_lost(

        original_df=
            original_df,

        resampled_df=
            four_ping_df,

        label=
            label
    )


    # -------------------------------------------------------------------------
    # General EDA of the new dataset
    # -------------------------------------------------------------------------

    four_ping_eda = summarize_fix_rate(

        four_ping_df,

        label=
            f"{label} four-pings-per-day"
    )


    # -------------------------------------------------------------------------
    # Specific four-pings-per-day verification
    # -------------------------------------------------------------------------

    (
        four_ping_daily_counts,
        four_ping_id_summary

    ) = evaluate_four_ping_output(

        four_ping_df,

        label=
            label
    )


    # -------------------------------------------------------------------------
    # Save output dataset
    # -------------------------------------------------------------------------

    output_file_name = (

        Path(file_name).stem

        +

        "_4pings_per_day_resampled.csv"
    )


    output_path = (

        OUTPUT_DIR /
        output_file_name
    )


    four_ping_df.to_csv(

        output_path,

        index=False
    )


    # -------------------------------------------------------------------------
    # Clean label for filenames
    # -------------------------------------------------------------------------

    safe_label = (

        label

        .replace(
            "(",
            ""
        )

        .replace(
            ")",
            ""
        )

        .strip()

        .lower()
    )


    # -------------------------------------------------------------------------
    # Retention table
    # -------------------------------------------------------------------------

    retention_path = (

        OUTPUT_DIR /

        f"{safe_label}"
        f"_4pings_retention_by_id.csv"
    )


    retention_table.to_csv(

        retention_path,

        index=False
    )


    # -------------------------------------------------------------------------
    # Daily target fix counts
    # -------------------------------------------------------------------------

    daily_counts_path = (

        OUTPUT_DIR /

        f"{safe_label}"
        f"_4pings_daily_fix_counts.csv"
    )


    four_ping_daily_counts.to_csv(

        daily_counts_path,

        index=False
    )


    # -------------------------------------------------------------------------
    # Per-ID sampling summary
    # -------------------------------------------------------------------------

    id_summary_path = (

        OUTPUT_DIR /

        f"{safe_label}"
        f"_4pings_id_summary.csv"
    )


    four_ping_id_summary.to_csv(

        id_summary_path,

        index=False
    )


    # -------------------------------------------------------------------------
    # Print saved paths
    # -------------------------------------------------------------------------

    print(
        "\nSaved four-pings-per-day dataset:"
    )

    print(
        output_path
    )


    print(
        "\nSaved supporting summaries:"
    )


    print(
        retention_path
    )


    print(
        daily_counts_path
    )


    print(
        id_summary_path
    )


    # -------------------------------------------------------------------------
    # Store for final combined comparison
    # -------------------------------------------------------------------------

    all_original_data[
        label
    ] = original_df


    all_four_ping_data[
        label
    ] = four_ping_df


    all_retention_tables[
        label
    ] = retention_table


# =============================================================================
# 10. COMBINED BEFORE-VERSUS-AFTER SUMMARY
# =============================================================================

comparison_rows = []


for label in DATASETS:


    original_df = (
        all_original_data[
            label
        ]
    )


    four_ping_df = (
        all_four_ping_data[
            label
        ]
    )


    # -------------------------------------------------------------------------
    # Original fixes per calendar day
    # -------------------------------------------------------------------------

    original_daily = (

        original_df

        .assign(

            calendar_date=

                original_df[
                    DATE_COL
                ]

                .dt.date
        )

        .groupby(
            [
                ID_COL,
                "calendar_date"
            ]
        )

        .size()
    )


    # -------------------------------------------------------------------------
    # Four-ping target day counts
    #
    # Use TARGET timestamp rather than actual observation date.
    # -------------------------------------------------------------------------

    four_ping_daily = (

        four_ping_df

        .assign(

            target_calendar_date=

                pd.to_datetime(

                    four_ping_df[
                        "target_four_ping_time"
                    ]

                ).dt.date
        )

        .groupby(
            [
                ID_COL,
                "target_calendar_date"
            ]
        )

        .size()
    )


    # -------------------------------------------------------------------------
    # Original intervals
    # -------------------------------------------------------------------------

    original_intervals = (

        original_df

        .sort_values(
            [
                ID_COL,
                DATE_COL
            ]
        )

        .groupby(ID_COL)[
            DATE_COL
        ]

        .diff()

        .dt.total_seconds()

        .div(3600)
    )


    # -------------------------------------------------------------------------
    # Actual intervals after four-ping resampling
    # -------------------------------------------------------------------------

    four_ping_intervals = (

        four_ping_df

        .sort_values(
            [
                ID_COL,
                DATE_COL
            ]
        )

        .groupby(ID_COL)[
            DATE_COL
        ]

        .diff()

        .dt.total_seconds()

        .div(3600)
    )


    # -------------------------------------------------------------------------
    # Original summary row
    # -------------------------------------------------------------------------

    comparison_rows.append({

        "dataset":
            label,

        "version":
            "Original approximately 3-hour",

        "target_pings_per_day":
            np.nan,

        "rows":
            len(original_df),

        "unique_ids":
            original_df[
                ID_COL
            ].nunique(),

        "median_interval_hours":
            original_intervals.median(),

        "mean_fixes_per_id_day":
            original_daily.mean(),

        "median_fixes_per_id_day":
            original_daily.median()
    })


    # -------------------------------------------------------------------------
    # Four-ping summary row
    # -------------------------------------------------------------------------

    comparison_rows.append({

        "dataset":
            label,

        "version":
            "Resampled to four pings/day (~6-hour)",

        "target_pings_per_day":
            PINGS_PER_DAY,

        "rows":
            len(four_ping_df),

        "unique_ids":
            four_ping_df[
                ID_COL
            ].nunique(),

        "median_interval_hours":
            four_ping_intervals.median(),

        "mean_fixes_per_id_day":
            four_ping_daily.mean(),

        "median_fixes_per_id_day":
            four_ping_daily.median()
    })


# =============================================================================
# 11. SAVE COMBINED COMPARISON
# =============================================================================

comparison_summary = pd.DataFrame(
    comparison_rows
)


comparison_output_path = (

    OUTPUT_DIR /

    "original_vs_4pings_per_day_sampling_summary.csv"
)


comparison_summary.to_csv(

    comparison_output_path,

    index=False
)


print(
    f"\n\n{'=' * 80}"
)

print(
    "FINAL ORIGINAL VERSUS FOUR-PINGS-PER-DAY COMPARISON"
)

print(
    f"{'=' * 80}"
)


print(

    comparison_summary

    .round(2)

    .to_string(
        index=False
    )
)


print(
    "\nSaved combined comparison:"
)

print(
    comparison_output_path
)


# =============================================================================
# 12. FINAL SANITY CHECK
# =============================================================================

print(
    f"\n{'=' * 80}"
)

print(
    "FINAL SAMPLING DESIGN CHECK"
)

print(
    f"{'=' * 80}"
)


print(

    f"Desired pings/day: "
    f"{PINGS_PER_DAY}"
)


print(

    f"Desired interval:  "
    f"{TARGET_INTERVAL_HOURS:.1f} hours"
)


if np.isclose(
    TARGET_INTERVAL_HOURS,
    6.0
):

    print(

        "\nPASS: "
        "Four pings per day correctly corresponds "
        "to six-hour sampling."
    )

else:

    raise AssertionError(

        "Sampling settings are inconsistent. "
        "Four pings/day should equal a "
        "six-hour interval."
    )


print(
    "\nProcessing complete."
)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 100% experiment: leak-free preprocessing with ID-disjoint splits + train-only fit
# ──────────────────────────────────────────────────────────────────────────────
import os, random
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

# ── REPRODUCIBILITY ───────────────────────────────────────────────────────────
os.environ["PYTHONHASHSEED"] = "0"
random.seed(13)
np.random.seed(13)

# ── CONFIG ─────────────────────────────────────────────────────────────────────
base       = Path("<LOCAL_PATH>")

# inputs: training pool is CWD-free; test is CWD-positive
train_raw  = base / "CWD_negative_training_with_seasons2_AKDE_FIXED_TRIMMED_150_4pings_per_day_resampled.csv"   # negatives only (still guard below)
test_raw   = base / "CWD_positive_testing_with_seasons2_AKDE_FIXED_TRIMMED_150_4pings_per_day_resampled.csv"     # positives (mixed OK; we only transform)

# outputs
out_train  = base / "train_processed_100.csv"
out_val    = base / "val_processed_100.csv"
out_test   = base / "test_processed_100.csv"

enc_path   = base / "encoder_100.pkl"
sca_path   = base / "scaler_100.pkl"
feat_path  = base / "feature_columns_100.pkl"
split_path = base / "split_ids_100.pkl"

to_drop = [
    'date_std', 'dx','dy','dt','burst','seg_key'
]

num_feats = [
    'velo','abs_rel_angle','sinuosity',
    'rolling_abs_rel_angle_mean','rolling_abs_rel_angle_sd',
    'rolling_abs_rel_angle_skew','rolling_abs_rel_angle_kurt',
    'rolling_velo_mean','rolling_velo_sd','rolling_velo_skew','rolling_velo_kurt',
    'akde_low','akde_est','akde_high'
]

cat_feats  = ['season']
meta_feats = ['id','date']

# ── UTILITIES ──────────────────────────────────────────────────────────────────
def basic_cleanup(df: pd.DataFrame) -> pd.DataFrame:
    """Drop unused cols, enforce numeric types, keep pkey intact, preserve order."""
    df = df.drop(columns=to_drop, errors='ignore').copy()
    
    # Preserve original datetime string AND a parsed version
    df['datetime_raw'] = df['date']   # keep raw untouched
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    
    for c in num_feats:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    
    df = df.dropna(subset=num_feats + cat_feats).copy()
    
    # Sort by id + parsed datetime (if valid), fallback to pkey for stability
    df = df.sort_values(['id', 'date']).reset_index(drop=True)
    return df

def transform_with_artifacts(df, encoder, scaler, feature_columns):
    """Apply fitted encoder/scaler; return DataFrame with pkey + meta + features."""
    cat_mat = encoder.transform(df[cat_feats])
    num_mat = scaler.transform(df[num_feats])
    cat_names = encoder.get_feature_names_out(cat_feats).tolist()
    X = np.hstack([num_mat, cat_mat])
    feat_df = pd.DataFrame(X, columns=num_feats + cat_names)

    feat_df = feat_df.reindex(columns=feature_columns, fill_value=0.0)
    out = pd.concat([df[meta_feats].reset_index(drop=True),
                     feat_df.reset_index(drop=True)], axis=1)
    return out

# ── 1) LOAD TRAIN POOL ────────────────────────────────────────────────────────
df_train_pool = pd.read_csv(train_raw)
df_train_pool = df_train_pool[df_train_pool['cwd_status'] == "negative"].copy()
df_train_pool = basic_cleanup(df_train_pool)

# ── 2) ID-DISJOINT SPLIT ──────────────────────────────────────────────────────
ids = df_train_pool['id'].unique()
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=13)
train_idx, val_idx = next(gss.split(df_train_pool, groups=df_train_pool['id']))

train_ids = df_train_pool.iloc[train_idx]['id'].unique()
val_ids   = df_train_pool.iloc[val_idx]['id'].unique()

df_train = df_train_pool[df_train_pool['id'].isin(train_ids)].copy()
df_val   = df_train_pool[df_train_pool['id'].isin(val_ids)].copy()

# ── 3) FIT ENCODER & SCALER ON TRAIN ONLY ─────────────────────────────────────
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
scaler  = MinMaxScaler()

encoder.fit(df_train[cat_feats])
scaler.fit(df_train[num_feats])

cat_names = encoder.get_feature_names_out(cat_feats).tolist()
feature_columns = num_feats + cat_names

# ── 4) TRANSFORM SPLITS ──────────────────────────────────────────────────────
df_train_out = transform_with_artifacts(df_train, encoder, scaler, feature_columns)
df_val_out   = transform_with_artifacts(df_val,   encoder, scaler, feature_columns)

# ── 5) TEST ───────────────────────────────────────────────────────────────────
df_test_raw = pd.read_csv(test_raw)
df_test_raw = basic_cleanup(df_test_raw)
df_test_out = transform_with_artifacts(df_test_raw, encoder, scaler, feature_columns)

# ── 6) SAVE ARTIFACTS ─────────────────────────────────────────────────────────
joblib.dump(encoder, enc_path)
joblib.dump(scaler,  sca_path)
joblib.dump(feature_columns, feat_path)

# always keep IDs as strings
train_ids_str = sorted({str(x).strip() for x in train_ids})
val_ids_str   = sorted({str(x).strip() for x in val_ids})
test_ids_str  = sorted({str(x).strip() for x in df_test_out['id'].astype(str).unique()})

joblib.dump(
    {"train_ids": train_ids_str,
     "val_ids":   val_ids_str,
     "test_ids":  test_ids_str},
    split_path
)

df_train_out.to_csv(out_train, index=False)
df_val_out.to_csv(out_val, index=False)
df_test_out.to_csv(out_test, index=False)

print("✅ Preprocessing complete with pkey preserved.")
print("Train IDs:", train_ids_str)
print("Val IDs:",   val_ids_str)
print("Test IDs:",  test_ids_str)




In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Autoencoder training USING CUSTOM LOSS + flexible thresholding
# ──────────────────────────────────────────────────────────────────────────────
import os, random
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# ===== Reproducibility =====
os.environ["PYTHONHASHSEED"] = "0"
random.seed(13); np.random.seed(13)
tf.keras.utils.set_random_seed(13)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

# ===== Config / Paths =====
base = "<LOCAL_PATH>"
train_path = f"{base}/train_processed_100.csv"
val_path   = f"{base}/val_processed_100.csv"   # negatives-only val (from preprocessing)
test_path  = f"{base}/test_processed_100.csv"

feat_cols_path = f"{base}/feature_columns_100.pkl"
model_path     = f"{base}/autoencoder_model.h5"
threshold_path = f"{base}/reconstruction_threshold.pkl"
history_path   = f"{base}/training_history.npy"
ckpt_path      = f"{base}/ae_ckpt.weights.h5"

# ===== Threshold policy (choose one) =====
THRESHOLD_POLICY = "train_lowest_p95"   # "train_lowest_p95" | "train_quantile" | "val_row_fpr" | "val_id_flag_rate" | "mad"

# Params for each policy
LOWEST_FRAC   = 0.75      # for "train_lowest_p95" 
LOWEST_Q      = 90        # percentile on the lowest fraction
TRAIN_Q       = 0.90      # for "train_quantile": e.g., 90th percentile of all train errors
VAL_ROW_FPR   = 0.20      # for "val_row_fpr": allow 20% of NEG val rows to flag → lower θ, more flags
VAL_ID_RATE   = 0.20      # for "val_id_flag_rate": allow 20% of NEG val IDs to flag at least 1 row
MIN_ROWS_PER_ID = 1       # ignore super-short IDs when computing ID-level rate
MAD_K         = 3.0       # for "mad"


# ===== Load features with frozen column order =====
feature_columns = joblib.load(feat_cols_path)

def load_X(path):
    df = pd.read_csv(path)
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    X = df[feature_columns].values.astype(np.float32)
    meta = df[["id"]].copy()
    if "datetime" in df.columns:
        meta["datetime"] = df["datetime"].values
    return X, meta, df

X_train, meta_train, df_train_raw = load_X(train_path)
X_val,   meta_val,   df_val_raw   = load_X(val_path)
X_test,  meta_test,  df_test_raw  = load_X(test_path)

# ===== tf.data pipelines =====
batch_size = 128
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(X, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, X))
    if training:
        ds = ds.shuffle(min(10000, len(X)), seed=13, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=False).cache().prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(X_train, training=True)
val_ds   = make_ds(X_val,   training=False)

# ===== Model =====
input_dim = X_train.shape[1]
reg = regularizers.l2(1e-4)

inp = layers.Input(shape=(input_dim,))
x = layers.Dense(32, activation='relu', kernel_regularizer=reg)(inp)
x = layers.Dense(16, activation='relu', kernel_regularizer=reg)(x)
z = layers.Dense(4,  activation='relu', kernel_regularizer=reg, name="bottleneck")(x)
y = layers.Dense(16, activation='relu', kernel_regularizer=reg)(z)
y = layers.Dense(32, activation='relu', kernel_regularizer=reg)(y)
out = layers.Dense(input_dim, activation='linear')(y)
autoencoder = Model(inp, out, name="autoencoder")

# === Custom loss ===
LAMBDA_DEV = 0.5                      # ← your choice
LOSS_SCALE = 1.0 + LAMBDA_DEV*input_dim  # α = 1 + λ d

def get_custom_loss(loss_scale=LOSS_SCALE):
    def custom_loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)  # mean over features
        return loss_scale * mse
    return custom_loss

loss_fn = get_custom_loss(LOSS_SCALE)
autoencoder.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=loss_fn)

# ===== Train =====
early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
ckpt = ModelCheckpoint(filepath=ckpt_path, save_weights_only=True, monitor="val_loss", save_best_only=True)

history = autoencoder.fit(
    train_ds, epochs=100, validation_data=val_ds, callbacks=[early_stop, ckpt], verbose=1
)

autoencoder.save(model_path)
np.save(history_path, history.history, allow_pickle=True)
print(f"✅ Saved model → {model_path}")

# ===== Errors on train/val (row-level, using SAME loss) =====
recon_train = autoencoder.predict(X_train, batch_size=batch_size, verbose=0)
train_errors = loss_fn(X_train, recon_train).numpy().astype(np.float64)

recon_val = autoencoder.predict(X_val, batch_size=batch_size, verbose=0)
val_errors = loss_fn(X_val, recon_val).numpy().astype(np.float64)

# ===== Threshold policies =====
def choose_threshold(policy: str):
    info = {"policy": policy}   # <— standard key

    if policy == "train_lowest_p95":
        n_low = int(len(train_errors) * LOWEST_FRAC)
        base = np.sort(train_errors)[:n_low]
        theta = float(np.percentile(base, LOWEST_Q))
        info.update({"lowest_frac": LOWEST_FRAC, "percentile": LOWEST_Q, "n_low": n_low})

    elif policy == "train_quantile":
        theta = float(np.quantile(train_errors, TRAIN_Q))
        info.update({"train_quantile": TRAIN_Q})

    elif policy == "val_row_fpr":
        q = 1.0 - float(VAL_ROW_FPR)
        theta = float(np.quantile(val_errors, q))
        info.update({"val_row_fpr": VAL_ROW_FPR, "val_quantile": q})

    elif policy == "val_id_flag_rate":
        q_grid = np.linspace(0.50, 0.999, 200)
        ids = df_val_raw["id"].values
        best_theta, best_rate, best_q = None, None, None
        for q in q_grid:
            th = np.quantile(val_errors, q)
            row_flag = val_errors > th
            id_any = pd.Series(row_flag).groupby(ids, sort=False).any()
            n_rows = pd.Series(1, index=ids).groupby(ids).sum()
            eligible = n_rows[n_rows >= MIN_ROWS_PER_ID].index
            rate = float(id_any.reindex(eligible).fillna(False).mean())
            if rate <= VAL_ID_RATE:
                best_theta, best_rate, best_q = float(th), rate, float(q)
                break
        if best_theta is None:
            best_q = float(q_grid[-1])
            best_theta = float(np.quantile(val_errors, best_q))
            row_flag = val_errors > best_theta
            id_any = pd.Series(row_flag).groupby(ids, sort=False).any()
            n_rows = pd.Series(1, index=ids).groupby(ids).sum()
            eligible = n_rows[n_rows >= MIN_ROWS_PER_ID].index
            best_rate = float(id_any.reindex(eligible).fillna(False).mean())
        theta = best_theta
        info.update({"val_id_rate_target": VAL_ID_RATE, "achieved_rate": best_rate, "val_quantile": best_q})

    elif policy == "mad":
        med = float(np.median(train_errors))
        mad = float(np.median(np.abs(train_errors - med)) + 1e-12)
        theta = med + MAD_K * 1.4826 * mad
        info.update({"median": med, "mad": mad, "k": MAD_K})

    else:
        raise ValueError(f"Unknown THRESHOLD_POLICY: {policy}")

    return theta, info

theta, th_info = choose_threshold(THRESHOLD_POLICY)

# Persist policy hyperparameters for reproducibility
if THRESHOLD_POLICY == "train_lowest_p95":
    th_info.update({"LOWEST_FRAC": float(LOWEST_FRAC), "LOWEST_Q": float(LOWEST_Q)})
elif THRESHOLD_POLICY == "train_quantile":
    th_info.update({"TRAIN_Q": float(TRAIN_Q)})
elif THRESHOLD_POLICY == "val_row_fpr":
    th_info.update({"VAL_ROW_FPR": float(VAL_ROW_FPR)})
elif THRESHOLD_POLICY == "val_id_flag_rate":
    th_info.update({"VAL_ID_RATE": float(VAL_ID_RATE), "MIN_ROWS_PER_ID": int(MIN_ROWS_PER_ID)})
elif THRESHOLD_POLICY == "mad":
    th_info.update({"MAD_K": float(MAD_K)})

th_info["effective_multiplier"] = float(LOSS_SCALE)  # alias for reviewers
print(f"[loss] λ={LAMBDA_DEV} | d={input_dim} | α=1+λ·d={LOSS_SCALE:.3f} (scaled-MSE)")


# ===== Diagnostics: how aggressive is θ? =====
def percent_rows_flagged(errors, theta):
    return float(np.mean(errors > theta))

train_row_rate = percent_rows_flagged(train_errors, theta)
val_row_rate   = percent_rows_flagged(val_errors, theta)

# ID-level on val (negatives)
val_ids = df_val_raw["id"].values
val_row_flag = val_errors > theta
val_id_any = pd.Series(val_row_flag).groupby(val_ids, sort=False).any()
val_n_rows = pd.Series(1, index=val_ids).groupby(val_ids).sum()
val_eligible = val_n_rows[val_n_rows >= MIN_ROWS_PER_ID].index
val_id_rate = float(val_id_any.reindex(val_eligible).fillna(False).mean())

print(f"📏 θ = {theta:.6g} | policy = {THRESHOLD_POLICY}")
print(f"   Train rows flagged: {100*train_row_rate:.1f}%")
print(f"   Val   rows flagged: {100*val_row_rate:.1f}%")
print(f"   Val     IDs flagged (≥1 row): {100*val_id_rate:.1f}% (min_rows={MIN_ROWS_PER_ID})")

# ===== Persist threshold + bookkeeping =====
# Diagnostics
# (unchanged, but print 'policy')
print(f"📏 θ = {theta:.6g} | policy = {THRESHOLD_POLICY}")

# Persist threshold + bookkeeping (use 'policy')
th_info.update({
    "policy": THRESHOLD_POLICY,
    "theta": float(theta),
    "lambda_dev": float(LAMBDA_DEV),
    "input_dim": int(input_dim),
    "loss_scale": float(LOSS_SCALE),  # α
    "loss_name": "scaled_mse_equivalent_to_mse_plus_lambda_sse",
    "train_rows_flagged": train_row_rate,
    "val_rows_flagged": val_row_rate,
    "val_ids_flagged": val_id_rate,
    "min_rows_per_id": int(MIN_ROWS_PER_ID)
})
joblib.dump(th_info, threshold_path)
print(f"✅ Threshold saved → {threshold_path}")


In [ ]:
print(df_train_raw.head(10))

In [ ]:
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from pathlib import Path

# ── CONFIG ─────────────────────────────────────────────────────────────────────
base = Path("<LOCAL_PATH>")
val_csv         = base / "val_processed_100.csv"
model_path      = base / "autoencoder_model.h5"
threshold_path  = base / "reconstruction_threshold.pkl"   # contains θ + loss params from training
feat_cols_path  = base / "feature_columns_100.pkl"      # enforce column order
param_out_path  = base / "id_rule_params.pkl"

USE_MIXED_VAL = False  # your val is negatives-only per preprocessing

# preregistered operating point (choose & freeze before test)
FROZEN_COUNT     = 75
FROZEN_FRAC      = 0.75
MIN_ROWS_PER_ID  = 250          # consider 100 if short IDs are noisy
AGGREGATION_RULE = "OR"       # freeze: "OR", "AND", "FRAC_ONLY", or "COUNT_ONLY"

# if you ever use mixed val with labels, these grids are considered a priori
COUNTS = [50, 100, 150, 200]
FRACS  = [0.80, 0.90, 0.925]
POS_IDS_VAL = []   # only used when USE_MIXED_VAL=True

# ── LOAD MODEL / FEATURES / TRAIN ARTIFACT ────────────────────────────────────
feature_columns = joblib.load(feat_cols_path)
autoencoder = load_model(model_path, compile=False)

# load training artifact with θ + loss params
th_info   = joblib.load(threshold_path)
theta     = float(th_info["theta"])
policy    = th_info.get("policy", "unknown")

# reconstruct scaled-MSE loss consistently
lambda_dev = float(th_info.get("lambda_dev", 0.5))
input_dim  = int(th_info.get("input_dim", len(feature_columns)))
loss_scale = float(th_info.get("loss_scale", 1.0 + lambda_dev * input_dim))

def get_custom_loss(loss_scale=loss_scale):
    def custom_loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
        return loss_scale * mse
    return custom_loss

loss_fn = get_custom_loss(loss_scale)
print(f"Using TRAIN-derived θ = {theta:.6f} (policy={policy}) | λ={lambda_dev} | d={input_dim} | α={loss_scale:.6g}")

# ── LOAD VAL AND COMPUTE ROW ERRORS (FROZEN FEATURE ORDER) ────────────────────
df_val = pd.read_csv(val_csv)
if "datetime" in df_val.columns:
    df_val["datetime"] = pd.to_datetime(df_val["datetime"], errors="coerce")

X_val     = df_val[feature_columns].values.astype(np.float32)
recon_val = autoencoder.predict(X_val, batch_size=128, verbose=0)
val_err   = loss_fn(X_val, recon_val).numpy().astype(np.float64)

rows_val = pd.DataFrame({
    "id": df_val["id"].values,
    "datetime": df_val["datetime"] if "datetime" in df_val.columns else pd.NaT,
    "reconstruction_error": val_err
})
rows_val["row_flag"] = rows_val["reconstruction_error"] > theta

# ── AGGREGATE BY ID ───────────────────────────────────────────────────────────
g         = rows_val.groupby("id", sort=False)
n_rows    = g.size()
n_flags   = g["row_flag"].sum()
flag_frac = n_flags / n_rows

# consistent eligible set (evaluation/selection must use the same gate)
eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index
n_rows    = n_rows.reindex(eligible_idx).astype(int)
n_flags   = n_flags.reindex(eligible_idx).astype(int)
flag_frac = flag_frac.reindex(eligible_idx).astype(float)

# ── SELECT (COUNT, FRAC) IF MIXED VAL; OTHERWISE FREEZE A PRIORI ─────────────
if USE_MIXED_VAL:
    pos_val = set(POS_IDS_VAL) & set(eligible_idx)
    if len(pos_val) == 0:
        raise ValueError("No positive IDs present in eligible validation set.")

    best = None; best_key = None
    for c in COUNTS:
        for f in FRACS:
            idx_count = set(n_flags[n_flags >= c].index)
            idx_frac  = set(flag_frac[flag_frac >= f].index)

            if   AGGREGATION_RULE == "AND":       pred = idx_count & idx_frac
            elif AGGREGATION_RULE == "FRAC_ONLY": pred = idx_frac
            elif AGGREGATION_RULE == "COUNT_ONLY":pred = idx_count
            else:                                 pred = idx_count | idx_frac  # OR

            recall = len(pred & pos_val) / len(pos_val)
            key = (recall, c, f)  # maximize recall; tie-break to stricter (higher c,f)
            if (best is None) or (key > best_key):
                best = {"count": int(c), "frac": float(f), "val_recall": float(recall)}
                best_key = key

    COUNT, FRAC = best["count"], best["frac"]
    print(f"Frozen ID rule from validation (positives-only, {AGGREGATION_RULE}): "
          f"COUNT={COUNT} | FRAC={FRAC} | recall={best['val_recall']:.3f}")
else:
    COUNT, FRAC = int(FROZEN_COUNT), float(FROZEN_FRAC)
    print(f"Using pre-registered ID rule ({AGGREGATION_RULE}; negatives-only val): "
          f"COUNT={COUNT} | FRAC={FRAC} | min_rows={MIN_ROWS_PER_ID}")

# ── SAVE FROZEN PARAMS FOR TEST ───────────────────────────────────────────────
frozen = {
    "theta": float(theta),
    "count": int(COUNT),
    "frac": float(FRAC),
    "lambda_dev": float(lambda_dev),
    "input_dim": int(input_dim),
    "loss_scale": float(loss_scale),
    "policy": policy,
    "aggregation_rule": AGGREGATION_RULE,  # ← frozen & explicit
    "val_meta": {
        "min_rows_per_id": int(MIN_ROWS_PER_ID),
        "use_mixed_val": bool(USE_MIXED_VAL)
    }
}
joblib.dump(frozen, param_out_path)
print(f"✅ Saved frozen params to: {param_out_path}")


In [ ]:
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from pathlib import Path

# ── PATHS ──────────────────────────────────────────────────────────────────────
base = Path("<LOCAL_PATH>")
test_feat_path  = base / "test_processed_100.csv"
model_path      = base / "autoencoder_model.h5"
frozen_param_p  = base / "id_rule_params.pkl"           # produced in validation (frozen params)
feat_cols_path  = base / "feature_columns_100.pkl"      # frozen feature order

rows_out_csv    = str(test_feat_path).replace(".csv", "_rows_with_flags_FROZEN.csv")
ids_out_csv     = str(test_feat_path).replace(".csv", "_id_summary_FROZEN.csv")
residuals_out   = str(test_feat_path).replace(".csv", "_rows_with_recon_and_residuals.csv")

# (Optional) test labels for evaluation
actual_cwd_instances_test     = ['A41', 'A43', 'A52', 'A56', 'A60', 'F106', 'F107', 'F26', 'F27', 'F30', 'F48', 'F62', 'F66', 'F73', 'F78', 'F79', 'F83', 'F87', 'F94']
unaffected_cwd_instances_test = []

# ── LOAD ARTIFACTS ────────────────────────────────────────────────────────────
feature_columns = joblib.load(feat_cols_path)           # list[str]
frozen = joblib.load(frozen_param_p)                    # dict from validation

theta  = float(frozen["theta"])
COUNT  = int(frozen["count"])
FRAC   = float(frozen["frac"])
policy = frozen.get("policy", "unknown")

# loss consistency (scaled-MSE, α = 1 + λ·d)
lambda_dev = float(frozen.get("lambda_dev", 0.5))
input_dim  = int(frozen.get("input_dim", len(feature_columns)))
loss_scale = float(frozen.get("loss_scale", 1.0 + lambda_dev * input_dim))

AGGREGATION_RULE = str(frozen.get("aggregation_rule", "OR")).upper()
MIN_ROWS_PER_ID  = int(frozen.get("val_meta", {}).get("min_rows_per_id", 1))

print(f"Using frozen params → θ={theta:.6f} | COUNT={COUNT} | FRAC={FRAC} | "
      f"λ={lambda_dev} | d={input_dim} | α={loss_scale:.3f} | MIN_ROWS={MIN_ROWS_PER_ID} | "
      f"policy={policy} | agg_rule={AGGREGATION_RULE}")

def get_custom_loss(loss_scale=loss_scale):
    def custom_loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
        return loss_scale * mse
    return custom_loss

loss_fn = get_custom_loss(loss_scale)
autoencoder = load_model(model_path, compile=False)

# ── LOAD TEST + COMPUTE RECONS + RESIDUALS ────────────────────────────────────
df = pd.read_csv(test_feat_path)
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

X_test = df[feature_columns].values.astype(np.float32)
recon  = autoencoder.predict(X_test, batch_size=128, verbose=0)

row_err = loss_fn(X_test, recon).numpy().astype(np.float64)
row_flag = row_err > theta

# absolute residuals per feature
residuals = np.abs(X_test - recon)
df_resid = pd.DataFrame(residuals, columns=[f"{c}_resid" for c in feature_columns])
df_recon = pd.DataFrame(recon, columns=[f"{c}_recon" for c in feature_columns])

rows_full = pd.concat([
    df.reset_index(drop=True)[["id", "date"]],
    pd.DataFrame({"reconstruction_error": row_err, "row_flag": row_flag}),
    df_recon, df_resid
], axis=1)

rows_full.to_csv(residuals_out, index=False)
print(f"✅ Saved per-row reconstructions + residuals → {residuals_out}")

# ── CONTINUE WITH AGGREGATION / IDS SUMMARY (unchanged) ───────────────────────
g        = pd.DataFrame({"row_flag": row_flag, "err": row_err, "id": df["id"]}).groupby("id", sort=False)
n_rows   = g.size()
n_flags  = g["row_flag"].sum()
flag_fr  = n_flags / n_rows
id_max   = g["err"].max()
id_mean  = g["err"].mean()

eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index
n_flags_elig = n_flags.reindex(eligible_idx)
flag_fr_elig = flag_fr.reindex(eligible_idx)

if AGGREGATION_RULE == "AND":
    pred_ids = set(n_flags_elig[n_flags_elig >= COUNT].index) & set(flag_fr_elig[flag_fr_elig >= FRAC].index)
elif AGGREGATION_RULE == "FRAC_ONLY":
    pred_ids = set(flag_fr_elig[flag_fr_elig >= FRAC].index)
elif AGGREGATION_RULE == "COUNT_ONLY":
    pred_ids = set(n_flags_elig[n_flags_elig >= COUNT].index)
else:  # OR
    pred_ids = set(n_flags_elig[n_flags_elig >= COUNT].index) | set(flag_fr_elig[flag_fr_elig >= FRAC].index)

pred_ids = sorted(pred_ids)
print("Predicted positive IDs:", pred_ids)

rows = pd.DataFrame({
    "id": df["id"].values,
    "date": df["date"] if "date" in df.columns else pd.NaT,
    "reconstruction_error": row_err,
    "row_flag": row_flag
})
rows.to_csv(rows_out_csv, index=False)


# Rule-aligned and component scores (for plots & AUC)
eps = 1e-12
count_norm_all = n_flags / max(COUNT, eps)
frac_norm_all  = flag_fr / max(FRAC, eps)

if AGGREGATION_RULE == "AND":
    score_all = np.minimum(count_norm_all, frac_norm_all)
elif AGGREGATION_RULE == "FRAC_ONLY":
    score_all = frac_norm_all
elif AGGREGATION_RULE == "COUNT_ONLY":
    score_all = count_norm_all
else:  # OR
    score_all = np.maximum(count_norm_all, frac_norm_all)

ids_df = pd.DataFrame({
    "id": n_rows.index,
    "n_rows": n_rows.values,
    "n_flags": n_flags.reindex(n_rows.index).values,
    "flag_frac": flag_fr.reindex(n_rows.index).values,
    "rule_aligned_score": score_all.reindex(n_rows.index).values,
    "frac_norm_score": frac_norm_all.reindex(n_rows.index).values,
    "count_norm_score": count_norm_all.reindex(n_rows.index).values,
    "id_max_error": id_max.reindex(n_rows.index).values,
    "id_mean_error": id_mean.reindex(n_rows.index).values,
    "eligible": [1 if i in set(eligible_idx) else 0 for i in n_rows.index],
    "is_positive_id": [1 if i in pred_ids else 0 for i in n_rows.index]
})
ids_df.to_csv(ids_out_csv, index=False)
print(f"✅ Wrote: {rows_out_csv}\n✅ Wrote: {ids_out_csv}")

# ── (OPTIONAL) TEST EVALUATION — eligible set only, rule-aligned score ────────
pos_set = set(actual_cwd_instances_test)
neg_set = set(unaffected_cwd_instances_test)

eval_ids   = sorted((pos_set | neg_set) & set(eligible_idx))
n_pos_eval = sum(i in pos_set for i in eval_ids)
n_neg_eval = sum(i in neg_set for i in eval_ids)
print(f"Eval on eligible IDs only → n_eval={len(eval_ids)} (pos={n_pos_eval}, neg={n_neg_eval}; min_rows={MIN_ROWS_PER_ID})")

if eval_ids and (n_pos_eval > 0 or n_neg_eval > 0):
    y_true = np.array([1 if i in pos_set else 0 for i in eval_ids])
    y_pred = np.array([1 if i in pred_ids else 0 for i in eval_ids])

    if (y_true == 1).all():
        TP = np.sum(y_pred == 1)
        FN = np.sum(y_pred == 0)
        FP = 0
        TN = 0

        sensitivity = TP / (TP + FN) if (TP + FN) else 0.0
        precision   = TP / (TP + FP) if (TP + FP) else 0.0
        accuracy    = (TP + TN) / (TP + FN + TN + FP) if (TP + FN + TN + FP) else 0.0
        f1_score    = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) else 0.0
        prevalence  = 1.0
        tss         = np.nan

        print(f"TP = {TP}, FN = {FN}")
        print(f"Sensitivity     (TPR) = {sensitivity:.3f}")
        print(f"Precision              = {precision:.3f}")
        print(f"Accuracy               = {accuracy:.3f}")
        print(f"F1 Score               = {f1_score:.3f}")
        print(f"Prevalence Identified  = {prevalence:.3f}")
        print(f"TSS                   = N/A (no negatives)")


In [ ]:
# === Overfitting sanity check: loss curves ===
import numpy as np
import matplotlib.pyplot as plt

hist = np.load(history_path, allow_pickle=True).item()  # {'loss': [...], 'val_loss': [...]}
train_loss = np.array(hist["loss"])
val_loss   = np.array(hist["val_loss"])

best_epoch = int(np.argmin(val_loss))
print("\n=== Loss history summary ===")
print(f"  Best val_loss epoch: {best_epoch} "
      f"(train_loss={train_loss[best_epoch]:.6g}, val_loss={val_loss[best_epoch]:.6g})")
print(f"  Final epoch train_loss: {train_loss[-1]:.6g}, val_loss: {val_loss[-1]:.6g}")
print(f"  Min train_loss overall: {train_loss.min():.6g}")
print(f"  Min val_loss   overall: {val_loss.min():.6g}")

plt.figure()
plt.plot(train_loss, label="train_loss")
plt.plot(val_loss,   label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (scaled MSE)")
plt.legend()
plt.title("Autoencoder train vs val loss")
plt.tight_layout()
plt.show()


In [ ]:
row_err = loss_fn(X_test, recon).numpy().astype(np.float64)
row_flag = row_err > theta
# Save a processed test file that includes row_flag (so contrib script can use it)
df_test_for_contrib = df.copy()
df_test_for_contrib["row_flag"] = row_flag.astype(bool)
df_test_for_contrib["reconstruction_error"] = row_err.astype(float)

PROC_TEST_WITH_FLAGS = str(test_feat_path).replace(".csv", "_WITH_FLAGS2.csv")
df_test_for_contrib.to_csv(PROC_TEST_WITH_FLAGS, index=False)
print("✅ PROC_TEST_WITH_FLAGS2 =", PROC_TEST_WITH_FLAGS)


In [ ]:
# ===== AE Feature Contributions — holdout-friendly (manuscript-ready, with TP/FN) =====
# UPDATED to plug directly into your AE testing run that already computed:
#   recon, pred_ids, MIN_ROWS_PER_ID, AGGREGATION_RULE, COUNT, FRAC,
#   plus your test id lists: actual_cwd_instances_test / unaffected_cwd_instances_test
#
# Key updates vs your original:
#   1) base/paths point to your Genotype folder (edit if needed)
#   2) PROC_TEST points to *_WITH_FLAGS.csv (written right after row_flag creation)
#   3) TRY_PREDICT_RECON = False (uses recon you already computed)
#   4) pos_set/neg_set + pred_ids injected so TP/FN partitions match frozen-rule outputs
#   5) removed duplicate function definitions near the bottom

import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
import joblib

# ----------------------------
# Config (point to your AE test run)
# ----------------------------
base   = Path("<LOCAL_PATH>")
outdir = base / "ae_holdout_results"
outdir.mkdir(parents=True, exist_ok=True)

# Processed test file MUST include row_flag (recommended: write *_WITH_FLAGS.csv after AE testing)
# If you followed the earlier step, you created:
#   PROC_TEST_WITH_FLAGS = str(test_feat_path).replace(".csv", "_WITH_FLAGS.csv")
# If you already have that variable in-session, this will use it; otherwise it falls back.
try:
    PROC_TEST = Path(PROC_TEST_WITH_FLAGS)  # created in AE testing step
except NameError:
    PROC_TEST = base / "test_processed_100_WITH_FLAGS.csv"  # rename if yours differs

SCA_PATH  = base / "scaler_100.pkl"             # optional; falls back to scaled units if missing
FEAT_PATH = base / "feature_columns_100.pkl"    # frozen feature order

# Use recon from upstream (AE testing)
TRY_PREDICT_RECON = False
BATCH = 256  # only used if TRY_PREDICT_RECON=True

# Row-level flagging (only used if PROC_TEST has no 'row_flag' column)
ROW_FLAG_TAU  = None
ROW_FLAG_PCTL = 95.0

# ID aggregation params (use frozen values from AE testing if present)
try:
    MIN_ROWS_PER_ID = int(MIN_ROWS_PER_ID)
    AGGREGATION_RULE = str(AGGREGATION_RULE).upper()
    COUNT = int(COUNT)
    FRAC  = float(FRAC)
except NameError:
    MIN_ROWS_PER_ID = 10
    AGGREGATION_RULE = "OR"
    COUNT = 5
    FRAC  = 0.05

# Ground-truth on holdout (prefer explicit lists from your AE testing script)
try:
    pos_set = set(actual_cwd_instances_test)
    neg_set = set(unaffected_cwd_instances_test)
    ASSUME_TEST_ALL_POS = (len(neg_set) == 0)
except NameError:
    ASSUME_TEST_ALL_POS = True
    pos_set, neg_set = None, None

# Prefer pred_ids computed from frozen rule (so partitions are identical)
try:
    pred_ids = sorted(set(pred_ids))
except NameError:
    pred_ids = None

# ----------------------------
# Load processed test & feature list
# ----------------------------
df_test    = pd.read_csv(PROC_TEST)
feat_names = joblib.load(FEAT_PATH)  # exact order used for training

missing_feats = [c for c in feat_names if c not in df_test.columns]
if missing_feats:
    raise KeyError(
        f"Processed test missing {len(missing_feats)} features. Example: {missing_feats[:10]}"
    )

X_test   = df_test[feat_names].to_numpy(dtype=np.float32)
ids_rows = df_test["id"].to_numpy()

has_row_flag = "row_flag" in df_test.columns

# ----------------------------
# Reconstructions & residuals
# ----------------------------
try:
    _ = recon  # use precomputed recon if provided
except NameError:
    if not TRY_PREDICT_RECON:
        raise ValueError("No 'recon' provided and TRY_PREDICT_RECON is False.")
    try:
        _ = autoencoder
    except NameError:
        raise ValueError("To predict 'recon' on the fly, define 'autoencoder' in the environment.")
    recon = autoencoder.predict(X_test, batch_size=BATCH, verbose=0)

recon = np.asarray(recon, float)
if X_test.shape != recon.shape:
    raise ValueError(f"Shape mismatch: X_test {X_test.shape} vs recon {recon.shape}")

residual_sq = (X_test - recon) ** 2     # (N_rows, n_features)
resid_total = residual_sq.sum(axis=1)   # per-row total residual (τ)

# ----------------------------
# Build the 'rows' frame for masks
# ----------------------------
rows = pd.DataFrame({"id": ids_rows})
if has_row_flag:
    rows["row_flag"] = df_test["row_flag"].astype(bool).to_numpy()
else:
    if ROW_FLAG_TAU is not None:
        rows["row_flag"] = resid_total > ROW_FLAG_TAU
    else:
        thr = np.nanpercentile(resid_total, ROW_FLAG_PCTL)
        rows["row_flag"] = resid_total >= thr

mask_all  = np.ones(len(rows), dtype=bool)
mask_flag = rows["row_flag"].to_numpy(bool)
mask_eval = mask_all.copy()  # whole holdout

# ----------------------------
# Per-feature contribution helper (→ %)
# ----------------------------
def contrib_pct(rsq, mask):
    if rsq.size == 0 or not np.any(mask):
        return np.full(rsq.shape[1], np.nan, dtype=float)
    m = np.nanmean(rsq[mask], axis=0)
    s = np.nansum(m)
    return (m / s) * 100.0 if s > 0 else np.full(rsq.shape[1], np.nan, dtype=float)

contrib_all   = contrib_pct(residual_sq, mask_all)
contrib_flag  = contrib_pct(residual_sq, mask_flag)
contrib_eval  = contrib_pct(residual_sq, mask_eval)

# ----------------------------
# Erraticness across IDs (median per-feature SD in %-points)
# ----------------------------
id_to_rows = defaultdict(list)
for r, i in enumerate(rows["id"].to_numpy()):
    id_to_rows[i].append(r)

def per_id_contrib_stats(mask):
    per_id = []
    for i, idxs in id_to_rows.items():
        idxs = np.asarray(idxs, dtype=int)
        idxs = idxs[mask[idxs]]
        if idxs.size == 0:
            continue
        m = np.nanmean(residual_sq[idxs], axis=0)
        s = np.nansum(m)
        per_id.append((i, (m / s * 100.0) if s > 0 else np.full(residual_sq.shape[1], np.nan)))
    if not per_id:
        return dict(median_feature_sd_pp=np.nan, n_ids=0)
    M = np.vstack([v for _, v in per_id])  # (n_ids, n_features)
    std_per_feat_pp = np.nanstd(M, axis=0)
    return dict(median_feature_sd_pp=float(np.nanmedian(std_per_feat_pp)), n_ids=len(per_id))

err_all  = per_id_contrib_stats(mask_all)
err_flag = per_id_contrib_stats(mask_flag)
err_eval = per_id_contrib_stats(mask_eval)

# ----------------------------
# Simple concentration proxies
# ----------------------------
def topk_share_pct(arr, k=3):
    a = np.asarray(arr, dtype=float)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return np.nan
    a_sorted = np.sort(a)[::-1]
    return float(np.sum(a_sorted[:min(k, a_sorted.size)]))

top3_all,  top5_all  = topk_share_pct(contrib_all, 3), topk_share_pct(contrib_all, 5)
top3_flag, top5_flag = topk_share_pct(contrib_flag, 3), topk_share_pct(contrib_flag, 5)
top3_eval, top5_eval = topk_share_pct(contrib_eval, 3), topk_share_pct(contrib_eval, 5)

# ----------------------------
# Per-feature table
# ----------------------------
df_contrib = pd.DataFrame({
    "feature":    feat_names,
    "All_%":      contrib_all,
    "Flagged_%":  contrib_flag,
    "Eval_%":     contrib_eval,
})
df_contrib["Delta_Flagged_minus_All_pp"] = df_contrib["Flagged_%"] - df_contrib["All_%"]

print("\n=== AE Feature Contributions (% of reconstruction error) — Holdout ===")
print(df_contrib.sort_values("Flagged_%", ascending=False).head(15).to_string(
    index=False, float_format=lambda x: f"{x:.2f}"
))

summary_tbl = pd.DataFrame([
    {"Group":"All rows",     "Top3_share_%": top3_all,  "Top5_share_%": top5_all,
     "Median_feat_SD_pp": err_all["median_feature_sd_pp"],  "n_ids": err_all["n_ids"]},
    {"Group":"Flagged rows", "Top3_share_%": top3_flag, "Top5_share_%": top5_flag,
     "Median_feat_SD_pp": err_flag["median_feature_sd_pp"], "n_ids": err_flag["n_ids"]},
    {"Group":"Eval IDs",     "Top3_share_%": top3_eval, "Top5_share_%": top5_eval,
     "Median_feat_SD_pp": err_eval["median_feature_sd_pp"], "n_ids": err_eval["n_ids"]},
])
summary_tbl["ΔFlagged−All_Top3_pp"] = summary_tbl.loc[1, "Top3_share_%"] - summary_tbl.loc[0, "Top3_share_%"]
summary_tbl["ΔFlagged−All_Top5_pp"] = summary_tbl.loc[1, "Top5_share_%"] - summary_tbl.loc[0, "Top5_share_%"]
summary_tbl["ΔFlagged−All_SD_pp"]   = summary_tbl.loc[1, "Median_feat_SD_pp"] - summary_tbl.loc[0, "Median_feat_SD_pp"]

print("\n=== Simple concentration & variability summary (holdout) ===")
print(summary_tbl.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# ----------------------------
# Category roll-up (movement story)
# ----------------------------
def categorize(fname: str) -> str:
    f = fname.lower()
    if "velo" in f: return "velocity"
    if "angle" in f: return "turning-angle"
    if "sinu" in f:  return "sinuosity"
    if "skew" in f or "kurt" in f: return "distributional-moment"
    if "rolling" in f or "mean" in f or "sd" in f: return "rolling-statistic"
    if f in {"low", "est", "high"} or "model" in f: return "model-derived"
    if "season" in f: return "season"
    return "other"

cat = pd.DataFrame({
    "feature":   feat_names,
    "category":  [categorize(f) for f in feat_names],
    "All_%":     contrib_all,
    "Flagged_%": contrib_flag,
    "Eval_%":    contrib_eval
})
cat_roll = (cat.groupby("category")[["All_%", "Flagged_%", "Eval_%"]]
              .sum()
              .sort_values("Flagged_%", ascending=False))
cat_roll["Delta_Flagged_minus_All_pp"] = cat_roll["Flagged_%"] - cat_roll["All_%"]

print("\n=== Category roll-up (% of recon error) — Holdout ===")
print(cat_roll.to_string(float_format=lambda x: f"{x:.1f}"))

# ----------------------------
# Build TP / FP / FN / TN partitions (ID-level)
# ----------------------------
# 1) Ground-truth ID sets (prefer injected pos_set/neg_set)
if (pos_set is None) or (neg_set is None):
    if (not ASSUME_TEST_ALL_POS) and ("CWD_label" in df_test.columns):
        id_lab = df_test.groupby("id")["CWD_label"].max()
        pos_set = set(id_lab.index[id_lab == 1])
        neg_set = set(id_lab.index[id_lab == 0])
    else:
        pos_set = set(df_test["id"].unique())
        neg_set = set()

# 2) Aggregate row_flags → predicted positive IDs (unless pred_ids already provided)
def aggregate_ids_from_rowflags(row_flags_bool, ids_series,
                                MIN_ROWS_PER_ID=MIN_ROWS_PER_ID,
                                AGGREGATION_RULE=AGGREGATION_RULE,
                                COUNT=COUNT, FRAC=FRAC):
    tmp = pd.DataFrame({"id": np.asarray(ids_series), "row_flag": row_flags_bool.astype(int)})
    g = tmp.groupby("id", sort=False)
    n_rows = g.size()
    n_flags = g["row_flag"].sum()
    flag_fr = n_flags / n_rows

    eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index
    n_flags_elig = n_flags.reindex(eligible_idx)
    flag_fr_elig = flag_fr.reindex(eligible_idx)

    if AGGREGATION_RULE == "AND":
        pred = set(n_flags_elig[n_flags_elig >= COUNT].index) & set(flag_fr_elig[flag_fr_elig >= FRAC].index)
    elif AGGREGATION_RULE == "FRAC_ONLY":
        pred = set(flag_fr_elig[flag_fr_elig >= FRAC].index)
    elif AGGREGATION_RULE == "COUNT_ONLY":
        pred = set(n_flags_elig[n_flags_elig >= COUNT].index)
    else:  # OR
        pred = set(n_flags_elig[n_flags_elig >= COUNT].index) | set(flag_fr_elig[flag_fr_elig >= FRAC].index)
    return sorted(pred), eligible_idx

if pred_ids is None:
    pred_ids, eligible_idx = aggregate_ids_from_rowflags(rows["row_flag"].to_numpy(bool), rows["id"].to_numpy())
else:
    # eligible IDs should follow the same MIN_ROWS rule; recompute eligibility only
    tmp = pd.DataFrame({"id": rows["id"].to_numpy()})
    n_rows = tmp.groupby("id", sort=False).size()
    eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index

# 3) Confusion partitions on eligible IDs
eval_ids = sorted((pos_set | neg_set) & set(eligible_idx))
TP_ids = [i for i in eval_ids if (i in pos_set) and (i in pred_ids)]
FP_ids = [i for i in eval_ids if (i in neg_set) and (i in pred_ids)]
FN_ids = [i for i in eval_ids if (i in pos_set) and (i not in pred_ids)]
TN_ids = [i for i in eval_ids if (i in neg_set) and (i not in pred_ids)]

rid = rows["id"]
row_flag_arr = rows["row_flag"].to_numpy(bool)

m_tp = rid.isin(TP_ids).to_numpy()
m_fp = rid.isin(FP_ids).to_numpy()
m_fn = rid.isin(FN_ids).to_numpy()
m_tn = rid.isin(TN_ids).to_numpy()

m_tp_flagged = m_tp & row_flag_arr
m_fp_flagged = m_fp & row_flag_arr
m_fn_unflagged = m_fn & (~row_flag_arr)
m_tn_unflagged = m_tn & (~row_flag_arr)

# ----------------------------
# Per-class contribution tables
# ----------------------------
def table_from_mask(mask, label):
    pct = contrib_pct(residual_sq, mask)
    return (pd.DataFrame({"feature": feat_names, f"{label}_%": pct})
              .sort_values(f"{label}_%", ascending=False)
              .reset_index(drop=True))

tp_tbl = table_from_mask(m_tp, "TP")
fp_tbl = table_from_mask(m_fp, "FP")
fn_tbl = table_from_mask(m_fn, "FN")
tn_tbl = table_from_mask(m_tn, "TN")

all_series = pd.Series(contrib_all, index=feat_names, name="All_%")

def top_with_delta(df_, label, k=10):
    col = f"{label}_%"
    if df_[col].notna().any():
        out = (df_.head(k)
                 .merge(all_series.reset_index().rename(columns={"index":"feature"}),
                        on="feature", how="left"))
        out[f"Delta_{label}_minus_All_pp"] = out[col] - out["All_%"]
        print(f"\n=== Top {k} features — {label} ===")
        print(out.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
    else:
        print(f"\n=== Top {k} features — {label} ===\n(no rows)")

top_with_delta(tp_tbl, "TP")
top_with_delta(fp_tbl, "FP")
top_with_delta(fn_tbl, "FN")
top_with_delta(tn_tbl, "TN")

# Save per-class CSVs
tp_tbl.to_csv(outdir / "ae_holdout_feat_contrib_tp.csv", index=False)
fp_tbl.to_csv(outdir / "ae_holdout_feat_contrib_fp.csv", index=False)
fn_tbl.to_csv(outdir / "ae_holdout_feat_contrib_fn.csv", index=False)
tn_tbl.to_csv(outdir / "ae_holdout_feat_contrib_tn.csv", index=False)

# Save ALL per-class features (merge All_% + delta for every feature)
for label, df_ in {"TP": tp_tbl, "FP": fp_tbl, "FN": fn_tbl, "TN": tn_tbl}.items():
    if df_ is not None and not df_.empty:
        merged = df_.merge(all_series.reset_index().rename(columns={"index": "feature"}),
                           on="feature", how="left")
        merged[f"Delta_{label}_minus_All_pp"] = merged[f"{label}_%"] - merged["All_%"]
        merged.to_csv(outdir / f"ae_all_features_{label}.csv", index=False)
        print(f"✅ Saved full feature list for {label}: {outdir}/ae_all_features_{label}.csv")

# ----------------------------
# Movement–residual linkage (inverse-scaling numeric features only)
# ----------------------------
try:
    scaler = joblib.load(SCA_PATH)
    if not hasattr(scaler, "inverse_transform"):
        raise TypeError("Loaded scaler has no inverse_transform")

    # Prefer the scaler's known feature names if present (skips one-hot/cats not seen by scaler)
    scaler_feats = list(getattr(scaler, "feature_names_in_", []))
    num_cols = [c for c in feat_names if c in scaler_feats] if scaler_feats else feat_names
    cat_cols = [c for c in feat_names if c not in num_cols]

    X_num = df_test[num_cols].to_numpy()
    X_cat = df_test[cat_cols].to_numpy() if cat_cols else np.empty((len(df_test), 0))
    X_num_raw = scaler.inverse_transform(X_num)

    df_raw = pd.DataFrame(np.hstack([X_num_raw, X_cat]), columns=num_cols + cat_cols)
    units_label = "raw units"
except Exception as e:
    print(f"[warn] Using scaled units — {e}")
    df_raw = df_test[feat_names].copy()
    units_label = "scaled units"

# Add IDs + row flags
df_raw["id"] = rows["id"].values
df_raw["row_flag"] = rows["row_flag"].values

# Collapsed movement metrics
df_raw["speed"]   = df_raw.filter(like="velo").mean(axis=1)
df_raw["turning"] = df_raw.filter(like="angle").mean(axis=1)
df_raw["sinu"]    = df_raw.filter(like="sinu").mean(axis=1)

def group_summary(mask, label):
    tmp = df_raw.loc[mask]
    return pd.Series({
        "n_rows":       int(len(tmp)),
        "mean_speed":   float(tmp["speed"].mean())   if len(tmp) else np.nan,
        "mean_turning": float(tmp["turning"].mean()) if len(tmp) else np.nan,
        "mean_sinu":    float(tmp["sinu"].mean())    if len(tmp) else np.nan
    }, name=label)

summ_pieces = [
    group_summary(mask_all, "All"),
    group_summary(mask_flag, "Flagged"),
]
if m_tp.any(): summ_pieces.append(group_summary(m_tp, "TP"))
if m_fn.any(): summ_pieces.append(group_summary(m_fn, "FN"))
if m_fp.any(): summ_pieces.append(group_summary(m_fp, "FP"))
if m_tn.any(): summ_pieces.append(group_summary(m_tn, "TN"))

summaries = pd.concat(summ_pieces, axis=1).T

# %diff vs TP (if TP exists)
if ("TP" in summaries.index) and (summaries.loc["TP", "n_rows"] > 0):
    for col in ["mean_speed", "mean_turning", "mean_sinu"]:
        base_val = summaries.loc["TP", col]
        for grp in summaries.index:
            if grp == "TP":
                continue
            summaries.loc[grp, f"{col}_vsTP_%diff"] = (
                (summaries.loc[grp, col] - base_val) / base_val * 100
                if (pd.notna(base_val) and base_val != 0) else np.nan
            )

print(f"\n=== Movement summaries by class ({units_label}) — Holdout ===")
print(summaries.round(3))
summaries.to_csv(outdir / f"ae_holdout_movement_summaries_by_class_{'raw' if units_label=='raw units' else 'scaled'}.csv",
                 index=True)

# Direct per-variable summaries
vars_of_interest = [
    "abs_rel_angle", "est", "low", "high",
    "rolling_abs_rel_angle_kurt", "rolling_abs_rel_angle_mean",
    "rolling_abs_rel_angle_sd", "rolling_abs_rel_angle_skew",
    "rolling_velo_kurt", "rolling_velo_mean", "rolling_velo_sd",
    "rolling_velo_skew", "sinuosity_28pt", "velo"
]
present = [v for v in vars_of_interest if v in df_raw.columns]

def group_summary_vars(mask, label):
    tmp = df_raw.loc[mask, present]
    stats = {f"{v}_mean": tmp[v].mean() for v in present}
    stats.update({f"{v}_sd": tmp[v].std() for v in present})
    stats["n_rows"] = len(tmp)
    return pd.Series(stats, name=label)

summ_vars_pieces = [
    group_summary_vars(mask_all,  "All"),
    group_summary_vars(mask_flag, "Flagged"),
]
if m_tp.any(): summ_vars_pieces.append(group_summary_vars(m_tp, "TP"))
if m_fn.any(): summ_vars_pieces.append(group_summary_vars(m_fn, "FN"))
if m_fp.any(): summ_vars_pieces.append(group_summary_vars(m_fp, "FP"))
if m_tn.any(): summ_vars_pieces.append(group_summary_vars(m_tn, "TN"))

summaries_vars = pd.concat(summ_vars_pieces, axis=1).T

# diffs vs TP (only if TP exists)
if "TP" in summaries_vars.index and summaries_vars.loc["TP", "n_rows"] > 0:
    for v in present:
        base = summaries_vars.loc["TP", f"{v}_mean"]
        for grp in summaries_vars.index:
            if grp == "TP":
                continue
            summaries_vars.loc[grp, f"{v}_diff_vsTP"] = summaries_vars.loc[grp, f"{v}_mean"] - base
            summaries_vars.loc[grp, f"{v}_pctdiff_vsTP"] = (
                (summaries_vars.loc[grp, f"{v}_mean"] - base) / base * 100
                if (pd.notna(base) and base != 0) else np.nan
            )

print("\n=== Per-variable movement summaries — Holdout ===")
print(summaries_vars.round(3))
summaries_vars.to_csv(outdir / f"ae_holdout_movement_summaries_by_variable_{'raw' if units_label=='raw units' else 'scaled'}.csv",
                      index=True)

# Residual correlation with movement metrics
movement_links = pd.DataFrame({
    "resid_total": resid_total,
    "speed":       df_raw["speed"],
    "turning":     df_raw["turning"],
    "sinu":        df_raw["sinu"],
}).corr()

print("\n=== Correlation of movement variables with total reconstruction error ===")
print(movement_links.round(3))
movement_links.to_csv(outdir / f"ae_holdout_movement_residual_correlations_{'raw' if units_label=='raw units' else 'scaled'}.csv",
                      index=True)

# ----------------------------
# Save the main contribution tables
# ----------------------------
(df_contrib.sort_values("Flagged_%", ascending=False)
          .to_csv(outdir / "ae_holdout_feature_contributions_by_feature.csv", index=False))
cat_roll.to_csv(outdir / "ae_holdout_feature_contributions_by_category.csv")
summary_tbl.to_csv(outdir / "ae_holdout_feature_contrib_summary_simple.csv", index=False)

print(f"\n💾 Saved tables to: {outdir}")
print(f"IDs — TP:{len(TP_ids)}  FN:{len(FN_ids)}  FP:{len(FP_ids)}  TN:{len(TN_ids)}  | Eligible:{len(set(eval_ids))}")


In [ ]:
# make_ae_feature_contributions_heatmap_BLUE.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path


# =============================================================================
# Global font settings
# =============================================================================

mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
})


# =============================================================================
# Paths
# =============================================================================

BASE = Path(
    r"<LOCAL_PATH>"
)

CSV_PATH = (
    BASE
    / "ae_holdout_results"
    / "ae_holdout_feature_contributions_by_feature.csv"
)

OUT_DIR = (
    BASE
    / "ae_holdout_results"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SAVE_BASE = (
    OUT_DIR
    / "ae_feature_contributions_heatmap_single_file"
)


# =============================================================================
# Model labels
#
# MODEL_LABEL is retained internally.
# MODEL_DISPLAY_LABEL controls only how the column heading appears in the figure.
# =============================================================================

MODEL_LABEL = (
    "White-Tailed Deer Autoencoder Model"
)

MODEL_DISPLAY_LABEL = (
    "Full Holdout"
)


# =============================================================================
# Load feature-contribution data
# =============================================================================

if not CSV_PATH.exists():

    raise FileNotFoundError(
        f"Input file not foun<LOCAL_PATH>"
        f"{CSV_PATH}"
    )


df = pd.read_csv(
    CSV_PATH
)


required_columns = {
    "feature",
    "All_%"
}


missing_columns = required_columns.difference(
    df.columns
)


if missing_columns:

    raise ValueError(
        f"Missing columns: "
        f"{sorted(missing_columns)}\n"
        f"Available columns: "
        f"{list(df.columns)}"
    )


df = df[
    [
        "feature",
        "All_%"
    ]
].copy()


df["All_%"] = pd.to_numeric(
    df["All_%"],
    errors="coerce"
)


df["model"] = MODEL_LABEL


# =============================================================================
# Convert to heatmap format
# =============================================================================

heatmap_df = df.pivot(
    index="feature",
    columns="model",
    values="All_%"
)


# =============================================================================
# Feature ordering and display labels
# =============================================================================

pretty_order = [

    # Velocity metrics
    "velo",
    "rolling_velo_mean",
    "rolling_velo_sd",
    "rolling_velo_skew",
    "rolling_velo_kurt",

    # Direction and turning-angle metrics
    "abs_rel_angle",
    "rolling_abs_rel_angle_mean",
    "rolling_abs_rel_angle_sd",
    "rolling_abs_rel_angle_skew",
    "rolling_abs_rel_angle_kurt",

    # Path-shape metrics
    "sinuosity",
    "sinuosity_28pt",
    "sinuosity_56pt",

    # Space-use metrics
    "akde_est",
    "akde_low",
    "akde_high"
]


pretty_map = {

    "velo":
        "Velocity",

    "rolling_velo_mean":
        "Rolling Velocity Mean",

    "rolling_velo_sd":
        "Rolling Velocity S.D.",

    "rolling_velo_skew":
        "Rolling Velocity Skew",

    "rolling_velo_kurt":
        "Rolling Velocity Kurtosis",

    "abs_rel_angle":
        "Absolute Relative Angle",

    "rolling_abs_rel_angle_mean":
        "Rolling Abs. Rel. Angle Mean",

    "rolling_abs_rel_angle_sd":
        "Rolling Abs. Rel. Angle S.D.",

    "rolling_abs_rel_angle_skew":
        "Rolling Abs. Rel. Angle Skew",

    "rolling_abs_rel_angle_kurt":
        "Rolling Abs. Rel. Angle Kurtosis",

    "sinuosity":
        "Sinuosity",

    "sinuosity_28pt":
        "Sinuosity",

    "sinuosity_56pt":
        "Sinuosity",

    "akde_est":
        "Median Space Usage",

    "akde_low":
        "Low C.I. Space Usage",

    "akde_high":
        "High C.I. Space Usage"
}


available_pretty_order = [
    feature
    for feature in pretty_order
    if feature in heatmap_df.index
]


heatmap_df = heatmap_df.reindex(
    available_pretty_order
)


heatmap_df.index = [
    pretty_map.get(
        feature,
        feature
    )
    for feature in heatmap_df.index
]


heatmap_df = heatmap_df.apply(
    pd.to_numeric,
    errors="coerce"
)


# =============================================================================
# Replace model column heading with publication-friendly display label
# =============================================================================

heatmap_df.columns = [
    MODEL_DISPLAY_LABEL
]


# =============================================================================
# Print numerical values for verification
# =============================================================================

print(
    "\nFeature-contribution table:"
)


print(
    heatmap_df
    .round(1)
    .to_string()
)


# =============================================================================
# Determine color scale
# =============================================================================

data_values = heatmap_df.to_numpy(
    dtype=float
)


if not np.isfinite(
    data_values
).any():

    raise ValueError(
        "The heatmap contains no finite "
        "feature-contribution values."
    )


true_min = float(
    np.nanmin(
        data_values
    )
)


true_max = float(
    np.nanmax(
        data_values
    )
)


vmin = 0.0


vmax = float(
    np.ceil(
        true_max
    )
)


if vmax <= vmin:

    vmax = 5.0


print(
    f"\nData range: "
    f"{true_min:.1f}% to "
    f"{true_max:.1f}%"
)


print(
    f"Plotting range: "
    f"{vmin:.0f}% to "
    f"{vmax:.0f}%"
)


# =============================================================================
# Build discrete blue color boundaries
#
# 0% to 5% receives a light blue.
# Larger feature contributions become progressively darker.
# =============================================================================

candidate_bounds = [
    vmin,
    5,
    10,
    15,
    20,
    30,
    40,
    50,
    vmax
]


bounds = [
    float(bound)
    for bound in candidate_bounds
    if vmin <= bound <= vmax
]


bounds.extend(
    [
        float(vmin),
        float(vmax)
    ]
)


bounds = sorted(
    set(bounds)
)


if len(bounds) < 2:

    bounds = [
        vmin,
        vmax
    ]


print(
    f"Color boundaries: "
    f"{bounds}"
)


# =============================================================================
# Create one blue shade for each discrete interval
#
# Avoid the nearly white lower portion of the Blues colormap.
# The highest interval receives the darkest blue.
# =============================================================================

base_cmap = mpl.colormaps[
    "Blues"
]


bin_midpoints = np.array(
    [
        (
            lower +
            upper
        ) / 2.0

        for lower, upper in zip(
            bounds[:-1],
            bounds[1:]
        )
    ],
    dtype=float
)


MIN_CMAP_POSITION = 0.15

MAX_CMAP_POSITION = 1.00


if vmax > 0:

    relative_positions = (
        bin_midpoints /
        vmax
    )

else:

    relative_positions = np.zeros_like(
        bin_midpoints
    )


relative_positions = np.clip(
    relative_positions,
    0,
    1
)


color_positions = (
    MIN_CMAP_POSITION
    +
    relative_positions
    *
    (
        MAX_CMAP_POSITION
        -
        MIN_CMAP_POSITION
    )
)


# =============================================================================
# Force first and last intervals to intended endpoints
# =============================================================================

if len(color_positions) > 0:

    color_positions[
        0
    ] = MIN_CMAP_POSITION

    color_positions[
        -1
    ] = MAX_CMAP_POSITION


discrete_colors = base_cmap(
    color_positions
)


discrete_cmap = mpl.colors.ListedColormap(
    discrete_colors,
    name="custom_discrete_blues"
)


norm = mpl.colors.BoundaryNorm(
    boundaries=bounds,
    ncolors=discrete_cmap.N,
    clip=True
)


print(
    "\nColor intervals:"
)


for index, (
    lower,
    upper
) in enumerate(
    zip(
        bounds[:-1],
        bounds[1:]
    )
):

    print(
        f"  {lower:>6.1f}% to "
        f"{upper:>6.1f}% "
        f"-> colormap position "
        f"{color_positions[index]:.3f}"
    )


# =============================================================================
# Figure dimensions
#
# Match the proportions of the CWD+ vs. CWD- heatmap.
# The overall figure remains wide enough for long feature names while the
# actual colored data column stays compact.
# =============================================================================

n_rows, n_cols = heatmap_df.shape


FIG_WIDTH = 8.0

ROW_HEIGHT = 0.62


FIG_HEIGHT = np.clip(
    ROW_HEIGHT * n_rows + 2.0,
    8.5,
    12.0
)


# =============================================================================
# Font sizes
# =============================================================================

AX_LABEL_FS = 26

X_TICK_FS = 19

Y_TICK_FS = 20

ANNOT_FS = 20

CBAR_LABEL_FS = 24

CBAR_TICK_FS = 18


# =============================================================================
# Create figure
#
# Fixed axes are used instead of constrained_layout so that both heatmaps
# have consistent column and colorbar proportions.
# =============================================================================

fig = plt.figure(
    figsize=(
        FIG_WIDTH,
        FIG_HEIGHT
    )
)


# =============================================================================
# Main heatmap axis
#
# [left, bottom, width, height]
# =============================================================================

ax = fig.add_axes(
    [
        0.52,   # left
        0.12,   # bottom
        0.18,   # width
        0.80    # height
    ]
)


# =============================================================================
# Colorbar axis
# =============================================================================

cax = fig.add_axes(
    [
        0.76,   # left
        0.12,   # bottom
        0.035,  # width
        0.80    # height
    ]
)


# =============================================================================
# Create heatmap
# =============================================================================

hm = sns.heatmap(
    heatmap_df,

    cmap=discrete_cmap,
    norm=norm,

    annot=True,
    fmt=".1f",

    annot_kws={
        "fontsize": ANNOT_FS,
        "fontfamily": "Times New Roman",
        "fontweight": "bold"
    },

    linewidths=1.0,
    linecolor="white",

    ax=ax,

    cbar=True,
    cbar_ax=cax
)


# =============================================================================
# Allow cell to use controlled axis width
# =============================================================================

ax.set_aspect(
    "auto"
)


# =============================================================================
# Determine darkest blue interval
# =============================================================================

darkest_threshold = (
    bounds[-2]
    if len(bounds) >= 2
    else vmax
)


# =============================================================================
# Force ALL annotations to the same Times New Roman bold style
# =============================================================================

flat_values = (
    heatmap_df
    .to_numpy()
    .flatten()
)


for text, value in zip(
    ax.texts,
    flat_values
):

    text.set_fontfamily(
        "Times New Roman"
    )

    text.set_fontsize(
        ANNOT_FS
    )

    text.set_fontweight(
        "bold"
    )


    if not np.isfinite(
        value
    ):

        continue


    # White text only on the darkest blue interval
    if value >= darkest_threshold:

        text.set_color(
            "white"
        )

    else:

        text.set_color(
            "black"
        )


# =============================================================================
# Axis labels
# =============================================================================

ax.set_xlabel(
    "Model Reconstruction",
    fontsize=AX_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold",
    labelpad=16
)


ax.set_ylabel(
    "Feature",
    fontsize=AX_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold",
    labelpad=18
)


# =============================================================================
# Tick positions
# =============================================================================

ax.tick_params(
    axis="x",
    rotation=0,
    length=0
)


ax.tick_params(
    axis="y",
    rotation=0,
    length=0
)


# =============================================================================
# Force x-axis labels to Times New Roman + bold
# =============================================================================

for label in ax.get_xticklabels():

    label.set_fontfamily(
        "Times New Roman"
    )

    label.set_fontsize(
        X_TICK_FS
    )

    label.set_fontweight(
        "bold"
    )

    label.set_rotation(
        0
    )

    label.set_horizontalalignment(
        "center"
    )

    label.set_multialignment(
        "center"
    )


# =============================================================================
# Force y-axis labels to Times New Roman + bold
# =============================================================================

for label in ax.get_yticklabels():

    label.set_fontfamily(
        "Times New Roman"
    )

    label.set_fontsize(
        Y_TICK_FS
    )

    label.set_fontweight(
        "bold"
    )

    label.set_rotation(
        0
    )


# =============================================================================
# Colorbar formatting
# =============================================================================

cbar = hm.collections[
    0
].colorbar


cbar.set_label(
    "% of Reconstruction Error",
    rotation=90,
    labelpad=18,
    fontsize=CBAR_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold"
)


# =============================================================================
# Colorbar ticks
# =============================================================================

candidate_ticks = np.array(
    [
        vmin,
        5,
        10,
        15,
        20,
        30,
        40,
        50,
        vmax
    ],
    dtype=float
)


nice_ticks = candidate_ticks[
    (
        candidate_ticks >= vmin
    )
    &
    (
        candidate_ticks <= vmax
    )
]


nice_ticks = np.unique(
    nice_ticks
)


cbar.set_ticks(
    nice_ticks
)


cbar.set_ticklabels(
    [
        f"{tick:.0f}%"
        for tick in nice_ticks
    ]
)


# =============================================================================
# Force all colorbar tick labels to same font and weight
# =============================================================================

for label in cbar.ax.get_yticklabels():

    label.set_fontfamily(
        "Times New Roman"
    )

    label.set_fontsize(
        CBAR_TICK_FS
    )

    label.set_fontweight(
        "bold"
    )


cbar.ax.tick_params(
    axis="y",
    length=5,
    width=1.3
)


cax.yaxis.set_label_position(
    "right"
)


cax.yaxis.set_ticks_position(
    "right"
)


# =============================================================================
# Strengthen colorbar outline
# =============================================================================

cbar.outline.set_linewidth(
    1.2
)


# =============================================================================
# Save and display
# =============================================================================

png_path = SAVE_BASE.with_suffix(
    ".png"
)


pdf_path = SAVE_BASE.with_suffix(
    ".pdf"
)


fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(
    pdf_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


print(
    f"\nSaved PNG to: "
    f"{png_path}"
)


print(
    f"Saved PDF to: "
    f"{pdf_path}"
)


plt.show()


plt.close(
    fig
)